In [ ]:
!pip install "numpy<2" scikit-learn matplotlib h5py hdf5plugin albumentations

In [ ]:
# IMPORTS

from datetime import datetime

import numpy as np 
import pandas as pd 
import os
import h5py
import hdf5plugin
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from albumentations import Compose, Rotate, HorizontalFlip, RandomBrightnessContrast, GaussNoise, Blur, Affine
from albumentations.pytorch import ToTensorV2

# Model
import torch.nn as nn

# Ground truth
from scipy.optimize import linear_sum_assignment
import torch.nn.functional as F

# Plotting
import matplotlib.pyplot as plt

# IoU score
import pandas as pd
from shapely.geometry import Polygon
from itertools import permutations
from math import isnan

# Validation
# from sklearn.model_selection import GroupKFold
from sklearn.model_selection import ShuffleSplit
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
import json
from tqdm import tqdm

# Testing
import torchvision.transforms.functional as VF

import torch.cuda.amp as amp

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# PRE-FIND AMOUNT OF DATA

start_time = datetime.now()


directory = '/kaggle/input/ae4353-y25'
file_lengths = []

filenames = [f for f in os.listdir(directory) if f.endswith('.h5') if f != 'test_set.h5']
filenames = sorted(filenames) # To ensure files in correct order

for name in filenames:
    with h5py.File(os.path.join(directory, name), 'r') as f:
        file_lengths.append(len(f["targets"]))

file_lengths[2] -= 1
# LittletonBlue is 142 but 141 to compensate


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time = time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

# file_lengths = [30, 56, 141, 195, 41, 36, 802, 807, 800, 804, 785, 784, 980, 983, 976, 980, 969, 978, 1912, 1948, 1888, 1947, 1917, 1912, 11830, 10278, 9713, 11870, 9077, 9541, 11634, 9511, 9903, 9602, 11662, 11276]
# filenames = ['BaltimoreMobile.h5', 'BaltimoreOB.h5', 'LittletonBlue.h5', 'LittletonHQ.h5', 'Warsaw1.h5', 'WashingtonOBNewLight.h5', 'autonomous_flight-01a-ellipse.h5', 'autonomous_flight-02a-ellipse.h5', 'autonomous_flight-03a-ellipse.h5', 'autonomous_flight-04a-ellipse.h5', 'autonomous_flight-05a-ellipse.h5', 'autonomous_flight-06a-ellipse.h5', 'autonomous_flight-07a-lemniscate.h5', 'autonomous_flight-08a-lemniscate.h5', 'autonomous_flight-09a-lemniscate.h5', 'autonomous_flight-10a-lemniscate.h5', 'autonomous_flight-11a-lemniscate.h5', 'autonomous_flight-12a-lemniscate.h5', 'autonomous_flight-13a-trackRATM.h5', 'autonomous_flight-14a-trackRATM.h5', 'autonomous_flight-15a-trackRATM.h5', 'autonomous_flight-16a-trackRATM.h5', 'autonomous_flight-17a-trackRATM.h5', 'autonomous_flight-18a-trackRATM.h5', 'piloted_flight-01p-ellipse.h5', 'piloted_flight-02p-ellipse.h5', 'piloted_flight-03p-ellipse.h5', 'piloted_flight-04p-ellipse.h5', 'piloted_flight-05p-ellipse.h5', 'piloted_flight-06p-ellipse.h5', 'piloted_flight-07p-lemniscate.h5', 'piloted_flight-08p-lemniscate.h5', 'piloted_flight-09p-lemniscate.h5', 'piloted_flight-10p-lemniscate.h5', 'piloted_flight-11p-lemniscate.h5', 'piloted_flight-12p-lemniscate.h5']

In [ ]:
# DATASET AND DATALOADER

start_time = datetime.now()


class CompDataset(Dataset):
    def __init__(self, directory='/kaggle/input/ae4353-y25', file_lengths=None, filenames=None, transforms=None, is_train=True, frame_rate=5, im_size=(320, 240), normalize_pixels=True):
        self.directory = directory
        self.filenames = filenames
        self.transforms = transforms
        self.is_train = is_train
        self.frame_rate = frame_rate
        self.im_size = im_size
        self.normalize_pixels = normalize_pixels

        if not self.is_train: # If test, don't need to do anything
            self.filenames = ['test_set.h5']
            self.cum_sizes = [95]
            self.video_ids = None

            return

        if self.filenames is None:
            self.filenames = [f for f in os.listdir(self.directory) if f.endswith('.h5') if f != 'test_set.h5']
            self.filenames = sorted(self.filenames) # So the files are always in the same order

        if file_lengths is None:
            file_lengths = []
            for name in self.filenames:
                with h5py.File(os.path.join(directory, name), 'r') as f:
                    file_lengths.append(len(f["targets"]))
            file_lengths[2] -= 1 # To compensate for LittletonBlue

        if self.frame_rate > 1: # Take only every nth frame since they are very similar
            self.cum_sizes = np.cumsum([(file_lengths[i]-1)//self.frame_rate + 1 if f.startswith(('autonomous', 'piloted')) else file_lengths[i] for (i,f) in enumerate(self.filenames)])

            self.video_ids = [file_idx for file_idx, length in enumerate(file_lengths) for _ in (range((length-1)//self.frame_rate + 1) if self.filenames[file_idx].startswith(('autonomous', 'piloted')) else range(length))]

        else:
            self.cum_sizes = np.cumsum(file_lengths)
            self.video_ids = [file_idx for file_idx, length in enumerate(file_lengths) for _ in range(length)]

    def __len__(self):
        return self.cum_sizes[-1]

    def __getitem__(self, idx):

        if self.is_train:
            file_idx = np.searchsorted(self.cum_sizes, idx, side='right')
            local_idx = idx - (self.cum_sizes[file_idx - 1] if file_idx > 0 else 0)
            local_idx = local_idx * self.frame_rate if (self.frame_rate > 1 and (self.filenames[file_idx].startswith(('autonomous', 'piloted')))) else local_idx

        else:
            file_idx = 0
            local_idx = idx

        # REMOVE FAULTY IMAGES IN LITTLETONBLUE
        if self.filenames[file_idx] == 'LittletonBlue.h5':
            local_idx += 1

        with h5py.File(os.path.join(self.directory, self.filenames[file_idx]), 'r') as f:
            image = f['images'][local_idx].transpose(1, 2, 0) # CHW --> HWC

            if self.is_train: # Keypoints (corners) and visibilities

                gates = f['targets'][str(local_idx).zfill(5)]

                # REMOVE FAULTY TARGETS IN LITTLETONBLUE
                if self.filenames[file_idx] == 'LittletonBlue.h5' and local_idx == 109:
                    gates = [gates[1]]

                keypoints = np.concatenate([g.reshape(4,3)[:, :2] for g in gates], axis=0) # [[x1,y1], [x2,y2],.., [x1,y1], [x2,y2],..]
                keypoints = keypoints*np.array(self.im_size, dtype=np.float32) # Find pixel index

                visibilities = np.array([(g[2::3]/2.0).astype(np.uint8) for g in gates]) # [[v1,v2,v3,v4], [v1,..],..]

            else:
                keypoints, visibilities = np.array([]), np.array([])

            if self.filenames[file_idx].startswith(('autonomous', 'piloted')): # Resize images (targets already resized)
                image = cv2.resize(image, self.im_size, interpolation=cv2.INTER_AREA)

            if self.is_train and self.transforms: # Transformations
                transformed = self.transforms(image=image, keypoints=keypoints)
                image = transformed['image']
                keypoints = transformed['keypoints']
                corners = np.array([keypoints[gn*4: (gn+1)*4] for gn in range(len(gates))]).round().astype(np.int64) # [[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..]]

            elif self.transforms:
                transformed = self.transforms(image=image)
                image = transformed['image']
                corners = np.array([])

            else:
                image = image.transpose(2,0,1)
                corners = np.array([keypoints[gn*4: (gn+1)*4] for gn in range(len(gates))]).round().astype(np.int64)

            if self.normalize_pixels:
                image = (image/255.0).to(torch.float32) if self.transforms else (image/255.0).astype(np.float32)

            return image, corners, visibilities


def collate_batch(batch):
    batch_images = torch.stack([sample[0] for sample in batch])
    batch_corners = [torch.tensor(sample[1]) for sample in batch]
    batch_visibilities = [torch.tensor(sample[2]) for sample in batch]
    return batch_images, batch_corners, batch_visibilities

file_lengths = [30, 56, 141, 195, 41, 36, 802, 807, 800, 804, 785, 784, 980, 983, 976, 980, 969, 978, 1912, 1948, 1888, 1947, 1917, 1912, 11830, 10278, 9713, 11870, 9077, 9541, 11634, 9511, 9903, 9602, 11662, 11276]
filenames = ['BaltimoreMobile.h5', 'BaltimoreOB.h5', 'LittletonBlue.h5', 'LittletonHQ.h5', 'Warsaw1.h5', 'WashingtonOBNewLight.h5', 'autonomous_flight-01a-ellipse.h5', 'autonomous_flight-02a-ellipse.h5', 'autonomous_flight-03a-ellipse.h5', 'autonomous_flight-04a-ellipse.h5', 'autonomous_flight-05a-ellipse.h5', 'autonomous_flight-06a-ellipse.h5', 'autonomous_flight-07a-lemniscate.h5', 'autonomous_flight-08a-lemniscate.h5', 'autonomous_flight-09a-lemniscate.h5', 'autonomous_flight-10a-lemniscate.h5', 'autonomous_flight-11a-lemniscate.h5', 'autonomous_flight-12a-lemniscate.h5', 'autonomous_flight-13a-trackRATM.h5', 'autonomous_flight-14a-trackRATM.h5', 'autonomous_flight-15a-trackRATM.h5', 'autonomous_flight-16a-trackRATM.h5', 'autonomous_flight-17a-trackRATM.h5', 'autonomous_flight-18a-trackRATM.h5', 'piloted_flight-01p-ellipse.h5', 'piloted_flight-02p-ellipse.h5', 'piloted_flight-03p-ellipse.h5', 'piloted_flight-04p-ellipse.h5', 'piloted_flight-05p-ellipse.h5', 'piloted_flight-06p-ellipse.h5', 'piloted_flight-07p-lemniscate.h5', 'piloted_flight-08p-lemniscate.h5', 'piloted_flight-09p-lemniscate.h5', 'piloted_flight-10p-lemniscate.h5', 'piloted_flight-11p-lemniscate.h5', 'piloted_flight-12p-lemniscate.h5']


end_time = datetime.now()
time_elapsed = end_time - start_time

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # VISUALIZE IMAGE AND TARGETS FROM DATASET, PREP

# print(len(train_ds))

# img1, c1, vis1 = train_ds[35]
# img = img1
# img1 = img1.numpy().transpose(1,2,0)

# # print(np.max(img1.numpy()))

# final_x = np.array([])
# final_y = np.array([])

# for g in c1: 
#     x1 = g[:, 0]
#     y1 = g[:, 1]
#     final_x = np.concatenate((final_x, x1))
#     final_y = np.concatenate((final_y, y1))


In [ ]:
# # TARGET FAULT DETECTION

# found1 = {}
# for f in filenames:
#     if not f.startswith(("autonomous", "piloted")):
#         for imn, i in enumerate(h5py.File(os.path.join('/kaggle/input/ae4353-y25', f), 'r')["targets"]):
#             count  = 0
#             for g in h5py.File(os.path.join('/kaggle/input/ae4353-y25', f), 'r')["targets"][i]:
#                 for x in g[::3]:
#                     if x > 1:
#                         print(f"x: {x}")
#                         count += 1
#                         found1[f+f"-{imn}"] = count
#                 for y in g[1::3]:
#                     if y > 1:
#                         print(f"y: {y}")
#                         count += 1
#                         found1[f+f"-{imn}"] = count
                    
# print(found1)

In [ ]:
# # VISUALIZE IMAGE AND ONE TARGET FAULT DETECTION, PREP

# # {'LittletonBlue.h5-0': 4, 'LittletonBlue.h5-109': 4, 'Warsaw1.h5-23': 1, 'Warsaw1.h5-30': 1}

# file = h5py.File('/kaggle/input/ae4353-y25/LittletonBlue.h5', 'r')
# print(file.keys())
# print(len(file["targets"]))

# # img_array = np.array(file["images"]).transpose(0,2,3,1)
# # img1 = img_array[0].transpose(1,2,0)

# img1 = file["images"][109].transpose(1, 2, 0)
# corners1 = file["targets"]["00109"][0]
# x1 = corners1[::3]
# y1 = corners1[1::3]
# x_coords = x1*len(img1[0])
# x_coords = np.round(x_coords).astype(int)
# y_coords = y1*len(img1)
# y_coords = np.round(y_coords).astype(int)


In [ ]:
# # VISUALIZE IMAGE AND TARGETS FROM .H5, PREP

# # Out of place: {'LittletonBlue.h5-0': 4, 'LittletonBlue.h5-109': 4, 'Warsaw1.h5-23': 1, 'Warsaw1.h5-30': 1}
# # Also: 'autonomous_flight-01a-ellipse.h5-501', aka image 1000, has 4 targets at (0,0) with visibility 0, and 4 correct targets.

# file = h5py.File('/kaggle/input/ae4353-y25/test_set.h5', 'r')
# print(file.keys())
# is_train = False


# # final_x = np.array([])
# # final_y = np.array([])

# # img_array = np.array(file["images"]).transpose(0,2,3,1)
# # img1 = img_array[0].transpose(1,2,0)

# img_number = 29
# # img_number = round((29/64)*len(file["images"]))+5
# img1 = file["images"][img_number]
# imgrot = VF.rotate(torch.tensor(img1), 15)
# imgrot2 = VF.rotate(imgrot, -15)
# # img_pad = pad_constant(torch.tensor(img1), 40)
# # imgrot_pad = VF.rotate(img_pad, 15)

# if is_train:
#     print(len(file["targets"]))
    
#     x_tl,x_tr,x_br,x_bl = [], [], [], []
#     y_tl, y_tr, y_br, y_bl = [], [], [] ,[]
    
#     visibilities = []
    
#     for g in file["targets"][str(img_number).zfill(5)]:
#         # x1 = g[::3]
#         # y1 = g[1::3]
#         # x_coords = x1*len(img1[0])
#         # x_coords = np.round(x_coords).astype(int)
#         # y_coords = y1*len(img1)
#         # y_coords = np.round(y_coords).astype(int)
#         # final_x = np.concatenate((final_x, x_coords))
#         # final_y = np.concatenate((final_y, y_coords))
    
#         x_tl.append(np.round(g[0]*len(img1[0])).astype(int))
#         x_tr.append(np.round(g[3]*len(img1[0])).astype(int))
#         x_br.append(np.round(g[6]*len(img1[0])).astype(int))
#         x_bl.append(np.round(g[9]*len(img1[0])).astype(int))
        
#         y_tl.append(np.round(g[1]*len(img1)).astype(int))
#         y_tr.append(np.round(g[4]*len(img1)).astype(int))
#         y_br.append(np.round(g[7]*len(img1)).astype(int))
#         y_bl.append(np.round(g[10]*len(img1)).astype(int))
    
#         visibilities.append(g[2::3])

In [ ]:
# # SINGLE IMAGE VISUALIZATION

# plt.imshow(img1.transpose(1, 2, 0))

# if is_train:

#     plt.scatter(
#         x_tl,
#         y_tl,
#         c='yellow',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_tr,
#         y_tr,
#         c='red',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_br,
#         y_br,
#         c='blue',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_bl,
#         y_bl,
#         c='white',
#         marker='o',
#         s=10
#     )
    
#     print(visibilities)

#     plt.show()

# # TL = yellow, TR = red, BR = blue, BL = white

In [ ]:
# # CHECK WHICH FILES ARE VIDEOS, MULTI-IMAGE VISUALIZATION --> All autonomous/piloted

# plt.subplots(4,4,figsize = (20,20))

# file = h5py.File('/kaggle/input/ae4353-y25/test_set.h5', 'r')
# start = 32

# for i in range(start, start+16):
#     img_np = file["images"][i].transpose(1, 2, 0)
#     plt.subplot(4,4,1+i-start)
#     plt.imshow(img_np)
# plt.show()

In [ ]:
# # VISUALIZE WHAT FRAME RATE IS GOOD

# train_transforms = Compose([
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=40)

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# print(len(train_ds))

# plt.subplots(4,4,figsize = (20,20))

# start = 3800

# for i in range(start, start+16):
#     img_np = train_ds[i][0].numpy().transpose(1, 2, 0)
#     plt.subplot(4,4,1+i-start)
#     plt.imshow(img_np)
# plt.show()

In [ ]:
# MODEL

start_time = datetime.now()


class DoubleConv(nn.Module):
    def __init__(self, input_ch, output_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_ch, output_ch, 3, padding=1),
            nn.BatchNorm2d(output_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(output_ch, output_ch, 3, padding=1),
            nn.BatchNorm2d(output_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, heatmap_ch=4, paf_ch=4):
        super().__init__()
        self.enc1 = DoubleConv(n_channels, 64) # Encode
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.enc5 = DoubleConv(512, 1024)

        self.pool = nn.MaxPool2d(2) # Downsample

        self.up5 = nn.ConvTranspose2d(1024, 512, 2, stride=2) # Upsample
        self.dec4 = DoubleConv(1024, 512) # Decode
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.heatmap_head = nn.Conv2d(64, heatmap_ch, 1)  # Condense information about single pixel into 4 channels, one for each corner. Heatmap about where the corner is most likely located.
        self.paf_head = nn.Conv2d(64, paf_ch * 2, 1)  # Part Affinity Fields --> 2d vector fields. For each pixel, compute a 2d vector that signals whether it is on a limb (line connecting 2 corners), and which direction it goes. One for each type of limb (connecting the 4 corners)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        e5 = self.enc5(self.pool(e4))

        d4 = self.up5(e5)
        d4 = torch.cat((d4, e4), dim=1) # Skip connections
        d4 = self.dec4(d4)
        d3 = self.up4(d4)
        d3 = torch.cat((d3, e3), dim=1)
        d3 = self.dec3(d3)
        d2 = self.up3(d3)
        d2 = torch.cat((d2, e2), dim=1)
        d2 = self.dec2(d2)
        d1 = self.up2(d2)
        d1 = torch.cat((d1, e1), dim=1)
        d1 = self.dec1(d1)

        heatmaps = torch.sigmoid(self.heatmap_head(d1))  # Sigmoid to transform into probability
        pafs = self.paf_head(d1)
        return heatmaps, pafs


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # GROUND TRUTH GENERATION, OLD VERSION

# start_time = datetime.now()


# def generate_gt_heatmaps(batch_corners, batch_visibilities, batch_size, device, img_height=240, img_width=320, sigma=3.0):
#     # batch_corners = (batch_size x n_gates x n_corners(4) x xy (2))
#     # batch_visibilities = (batch_size x n_gates x n_corners(4))
#     heatmaps = torch.zeros((batch_size, 4, img_height, img_width), dtype=torch.float32, device=device)
#     for sample,(corners, visibilities) in enumerate(zip(batch_corners, batch_visibilities)):
#         for gate_corners, gate_vis in zip(corners, visibilities):
#             for c, (xy, vis) in enumerate(zip(gate_corners, gate_vis)):
#                 # Top left, top right, bottom right, bottom left
#                 if vis:
#                     x, y = xy[0], xy[1]

#                     window = round(6 * sigma) # 6*sigma = 99%
#                     x_start = max(0, x - window//2)
#                     x_end = min(img_width, x + window//2 + 1)
#                     y_start = max(0, y - window//2)
#                     y_end = min(img_height, y + window//2 + 1)

#                     if x_start >= x_end or y_start >= y_end:
#                     # If point outside of bounds more than window reaches, no heatmap
#                         continue

#                     # Compute a gaussian distribution around the target point
#                     Y, X = torch.meshgrid(torch.arange(y_start, y_end, dtype=torch.float32, device=device),
#                                           torch.arange(x_start, x_end, dtype=torch.float32, device=device),
#                                           indexing='ij') # Make a grid of the window range. I.e. by accessing the Y and X grid with the same indexes you would access the window, now you get the y and x coordinate for that pixel.
#                     dist_sq = (X - x)**2 + (Y - y)**2 # Squared distance from center
#                     gaussian = torch.exp(-dist_sq / (2.0 * sigma**2)) # Gaussian distribution centered on (x,y)

#                     heatmaps[sample, c, y_start:y_end, x_start:x_end] = torch.maximum(heatmaps[sample, c, y_start:y_end, x_start:x_end], gaussian)

#     return heatmaps
    

# def generate_gt_pafs(batch_corners, batch_visibilities, batch_size, device, img_height=240, img_width=320, limb_width=2.0):
#     # batch_corners = (batch_size x n_gates x n_corners(4) x xy (2))
#     # batch_visibilities = (batch_size x n_gates x n_corners(4))
#     pafs = torch.zeros((batch_size, 8, img_height, img_width), dtype=torch.float32, device=device)
#     limbs = [(0,1), (1,2), (0,3), (3,2)] # TL-TR, TR-BR, TL-BL, BL-BR

#     for sample, (corners, visibilities) in enumerate(zip(batch_corners, batch_visibilities)):
#         count = torch.zeros((4, img_height, img_width), dtype=torch.float32, device=device)  # To count occurences in same pixel, i.e. limb overlap
#         for gate_corners, gate_vis in zip(corners, visibilities):
#             for l, (c1, c2) in enumerate(limbs):
#                 if gate_vis[c1] or gate_vis[c2]: # If either corner is visible, the limb is visible
#                     xy1 = gate_corners[c1].to(device)
#                     xy2 = gate_corners[c2].to(device)
#                     vec = xy2 - xy1

#                     magnitude = torch.sqrt((vec**2).sum())
#                     line_length = max(abs(vec[0].item()), abs(vec[1].item())) + 1 # DDA algorithm
#                     if line_length == 0:
#                         continue

#                     unit_vec = vec/magnitude
#                     # Sample points along line
#                     t = torch.linspace(0, 1, line_length, dtype=torch.float32, device=device)
#                     points = (xy1 + t.reshape(line_length, 1)*vec).round().to(torch.int64)
#                     for p in points:
#                         cx, cy = p[0].item(), p[1].item()
#                         x_start = max(0, round(cx - limb_width))
#                         x_end = min(img_width, round(cx + limb_width + 1))
#                         y_start = max(0, round(cy - limb_width))
#                         y_end = min(img_height, round(cy + limb_width + 1))

#                         if x_start >= x_end or y_start >= y_end:
#                             continue

#                         Y, X = torch.meshgrid(torch.arange(y_start, y_end, dtype=torch.float32, device=device),
#                                               torch.arange(x_start, x_end, dtype=torch.float32, device=device),
#                                               indexing='ij')
#                         dx = X - cx # Limb line center to surrounding point
#                         dy = Y - cy
#                         proj = dx * unit_vec[0] + dy * unit_vec[1] # Projection of (dx, dy) onto limb line
#                         perp_dist_sq = dx**2 + dy**2 - proj**2 # Pythagoras
#                         mask = perp_dist_sq <= limb_width**2 # Only keep pixels within limb_width of center
#                         # Add unit vectors, then average by dividing occurences across gates, if there are overlaps
#                         pafs[sample, 2*l, y_start:y_end, x_start:x_end][mask] += unit_vec[0] # Within limb, x component of unit-vector
#                         pafs[sample, 2*l + 1, y_start:y_end, x_start:x_end][mask] += unit_vec[1] # y component of unit-vector
#                         count[l, y_start:y_end, x_start:x_end][mask] += 1.0

#         for l in range(4): # Average pafs by vector count
#             mask = count[l] > 0 # Only pixels that appeared more than once
#             pafs[sample, 2*l][mask] /= count[l][mask]
#             pafs[sample, 2*l + 1][mask] /= count[l][mask]

#     return pafs


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Time elapsed: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# GROUND TRUTH GENERATION (VECTORIZED BY IMAGE)

start_time = datetime.now()


def generate_gt_heatmaps(batch_corners, batch_vis, img_size, device, sigma=3.0):

    # batch_corners = (batch_length x n_gates x n_corners(4) x xy(2))
    # batch_vis = (batch_length x n_gates x n_corners(4))
    # img_size = (240, 320)

    batch_length = len(batch_corners)
    batch_heatmaps = torch.zeros((batch_length, 4, *img_size), dtype=torch.float32, device=device)
    window_size = round(6 * sigma) # 6*sigma = 99%, window of gaussian around each point
    
    Y, X = torch.meshgrid(torch.arange(img_size[0], device=device),
                          torch.arange(img_size[1], device=device),
                          indexing='ij') # Make a grid of the image. I.e. Y[y][x] you get the y-coordinate (which is y), X[y][x] you get the x-coordinate (which is x)

    for img in range(batch_length):
        n_gates = len(batch_corners[img]) # n_gates in that image
        if n_gates == 0:
            continue

        corners = batch_corners[img].to(device)
        visibilities = batch_vis[img].to(device)

        x_flat = corners.view(-1, 2)[:, 0] # (n_gates*4, 2)[:,0] --> [x_1, x_2,.. x_(n_gates*4)]
        y_flat = corners.view(-1, 2)[:, 1] # [y_1, y_2,.. y_(n_gates*4)]
        v_flat = visibilities.view(-1) # [v_1, v_2,.. v_(n_gates*4)]
        corner_types = torch.arange(4, device=device).tile(n_gates) # [0, 1, 2, 3, 0, 1, 2, 3,..] (n_gates times)

        mask = v_flat > 0 # Only select xs, ys and their corner type that are visible
        if not mask.any():
            continue

        x_flat, y_flat, corner_types = x_flat[mask], y_flat[mask], corner_types[mask]

        for i in range(len(x_flat)): # For each of the visible points

            # Find the window for the gaussian around the point
            x_start = max(0, int(x_flat[i] - window_size//2))
            x_end = min(img_size[1], int(x_flat[i] + window_size//2 + 1))
            y_start = max(0, int(y_flat[i] - window_size//2))
            y_end = min(img_size[0], int(y_flat[i] + window_size//2 + 1))

            if x_start >= x_end or y_start >= y_end:
                # If point outside of bounds more than window reaches, no heatmap
                continue

            # Compute a gaussian distribution around the target point within the window
            dist_sq = (X[y_start:y_end, x_start:x_end] - x_flat[i])**2 + (Y[y_start:y_end, x_start:x_end] - y_flat[i])**2 # For each point/pixel in the window, compute distance squared to the corner point
            gaussian = torch.exp(-dist_sq / (2.0 * sigma**2)) # Gaussian distribution centered on (x,y), with max value 1.0 and within window
            # For each corner, heatmap within that window is max of existing heatmap and found gaussian
            batch_heatmaps[img, int(corner_types[i]), y_start:y_end, x_start:x_end] = torch.maximum(batch_heatmaps[img, int(corner_types[i]), y_start:y_end, x_start:x_end], gaussian)

    return batch_heatmaps


def generate_gt_pafs(batch_corners, batch_vis, img_size, device, limb_width=2.0):

    # batch_corners = (batch_length x n_gates x n_corners(4) x xy(2))
    # batch_vis = (batch_length x n_gates x n_corners(4))
    # img_size = (240, 320)

    batch_length = len(batch_corners)
    batch_pafs = torch.zeros((batch_length, 8, *img_size), dtype=torch.float32, device=device)
    limbs = torch.tensor([(0,1), (1,2), (0,3), (3,2)], device=device)  # TL-TR, TR-BR, TL-BL, BL-BR
    sample_rate = limb_width*2 + 1

    for img in range(batch_length):
        n_gates = len(batch_corners[img])
        if n_gates == 0:
            continue

        corners = batch_corners[img].to(device) # (n_gates x n_corners(4) x xy(2))
        visibilities = batch_vis[img].to(device) # (n_gates x n_corners(4))

        xy1 = corners[:, limbs[:,0]]  # (n_gates x n_corners(4) x xy(2)) limb start points, basically re-ordered "corners"
        xy2 = corners[:, limbs[:,1]]  # (n_gates x n_corners(4) x xy(2)) limb end points
        limb_vis = visibilities[:, limbs[:,0]] * visibilities[:, limbs[:,1]]  # (n_gates x n_corners(4)), i.e. limb only visible if both corners visible
        limb_types = torch.arange(4, device=device).tile(n_gates).reshape(-1, 4) # [[0, 1, 2, 3], [0, 1, 2, 3],..] (n_gates times)

        mask = limb_vis > 0 # Only select points that are visible
        if not mask.any():
            continue

        xy1, xy2, limb_types = xy1[mask], xy2[mask], limb_types[mask] # (n_visible_limbs x xy(2)) and (n_visible_limbs)
        vecs = xy2 - xy1
        magnitudes = torch.linalg.vector_norm(vecs.to(torch.float32), dim=1) # (n_visible_limbs)
        line_lengths = torch.max(torch.abs(vecs), dim=1).values + 1 # DDA algorithm

        valid = line_lengths > 0 # Only vectors longer than 0
        vecs = vecs[valid] # (n_valid_vecs x xy(2))
        xy1 = xy1[valid] # (n_valid_vecs x xy(2))
        limb_types = limb_types[valid] # (n_valid_vecs)
        magnitudes = magnitudes[valid] # (n_valid_vecs)
        line_lengths = line_lengths[valid] # (n_valid_vecs)

        unit_vecs = vecs / magnitudes.unsqueeze(1) # (n_valid_vecs x xy(2))
   
        num_samples = torch.floor((line_lengths-1)/sample_rate).to(torch.int32) + ((line_lengths-1)%sample_rate != 0).to(torch.int32) + 1 # (n_valid_vecs). Only every (limb_width*2 + 1)th sample since the in-between are covered by limb width. Include start and end points.

        t = torch.cat([torch.linspace(0, 1, int(ns), device=device) for ns in num_samples], dim=0) # (n_valid_vecs*n_samples)
        points = xy1.repeat_interleave(num_samples, dim=0) + t.unsqueeze(1) * vecs.repeat_interleave(num_samples, dim=0)  # (n_valid_vecs*n_samples x xy(2)) +  (n_valid_vecs*n_samples x 1) * (n_valid_vecs*n_samples x xy(2)) --> (n_valid_vecs*n_samples x xy(2))
        sample_to_vec = torch.arange(len(num_samples), device=device, dtype=torch.int32).repeat_interleave(num_samples, dim=0) # (n_valid_vecs*n_samples)

        count = torch.zeros((4, *img_size), device=device) # (4, 240, 320). To count occurences in same pixel, i.e. limb overlap, per limb type

        starts = torch.maximum(torch.tensor([0], device=device), torch.round(points - limb_width).to(torch.int32)) # (n_valid_vecs*n_samples x xy(2))
        ends = torch.minimum(torch.tensor([[img_size[1], img_size[0]]], device=device).to(torch.int32), torch.round(points + limb_width + 1).to(torch.int32)) # (n_valid_vecs*n_samples x xy(2))

        valid_idxs = starts < ends # (n_valid_vecs*n_samples x 2)
        valid_idxs = valid_idxs.all(dim=1) # (n_valid_vecs*n_samples)
        starts, ends = starts[valid_idxs], ends[valid_idxs] # (n_valid_sample_points x xy(2)). Only keep start and end x,y coordinates for those where both x_start < x_end and y_start < y_end
        sample_to_vec = sample_to_vec[valid_idxs] # (n_valid_sample_points)

        for i, v in enumerate(sample_to_vec):
            l = limb_types[v] # Is it limb type 0, 1, 2 or 3?

            # Add unit vectors, then average by dividing occurences across gates, if there are overlaps
            batch_pafs[img, 2*l, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += unit_vecs[v, 0] # Within limb, x component of unit-vector
            batch_pafs[img, 2*l + 1, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += unit_vecs[v, 1] # y component of unit-vector
            count[l, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += 1.0

        for l in range(4):
            mask_c = count[l] > 1.0 # If a pixel occurred more than once
            batch_pafs[img, l*2][mask_c] /= count[l][mask_c] # Divide paf value by its amount of occurences
            batch_pafs[img, l*2 + 1][mask_c] /= count[l][mask_c]

    return batch_pafs


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # FULLY VECTORIZED VERSION OF GROUND TRUTH GENERATION

# start_time = datetime.now()


# def generate_gt_heatmaps_vectorized(batch_corners, batch_vis, img_size, device, sigma=3.0):

#     # batch_corners = (batch_length x n_gates x n_corners(4) x xy(2))
#     # batch_vis = (batch_length x n_gates x n_corners(4))
#     # img_size = (240, 320)

#     batch_length = len(batch_corners)
#     batch_heatmaps = torch.zeros((batch_length, 4, *img_size), dtype=torch.float32, device=device)
#     window_size = round(6 * sigma) # 6*sigma = 99%, window of gaussian around each point

#     gates_per_image = torch.tensor([len(corners) for corners in batch_corners], dtype=torch.int32, device=device)
#     mask_no_gates = gates_per_image > 0 
    
#     corners = torch.concat([corners.to(device).view(-1, 2) for img,corners in enumerate(batch_corners) if mask_no_gates[img]], dim=0) # (n_gates*4*batch_length, 2)
#     x_flat = corners[:, 0] # [x_1, x_2,.. x_(n_gates*4*batch_length)]
#     y_flat = corners[:, 1] # [y_1, y_2,.. y_(n_gates*4*batch_length)]
#     v_flat = torch.concat([visibilities.to(device).view(-1) for img,visibilities in enumerate(batch_vis) if mask_no_gates[img]], dim=0) # [v_1, v_2,.. v_(n_gates*4*batch_length)]
#     corner_types = torch.arange(4, device=device).tile(torch.sum(gates_per_image)) # [0, 1, 2, 3, 0, 1, 2, 3,..] (n_gates*batch_length times)
#     img_number = torch.arange(batch_length, device=device).repeat_interleave(gates_per_image*4, dim=0)

#     mask = v_flat > 0 # Only select xs, ys and their corner type and img number that are visible
#     if not mask.any():
#         return batch_heatmaps

#     x_flat, y_flat, corner_types, img_number = x_flat[mask], y_flat[mask], corner_types[mask], img_number[mask]
#     Y, X = torch.meshgrid(torch.arange(img_size[0], device=device),
#                           torch.arange(img_size[1], device=device),
#                           indexing='ij') # Make a grid of the image. I.e. Y[y][x] you get the y-coordinate (which is y), X[y][x] you get the x-coordinate (which is x)

#     x_starts = torch.maximum(torch.tensor(0, device=device), torch.round(x_flat - window_size//2).to(torch.int32)) # (n_vis_points)
#     y_starts = torch.maximum(torch.tensor(0, device=device), torch.round(y_flat - window_size//2).to(torch.int32)) # (n_vis_points)
#     x_ends = torch.minimum(torch.tensor(img_size[1], device=device).to(torch.int32), torch.round(x_flat + window_size//2 + 1).to(torch.int32)) # (n_vis_points)
#     y_ends = torch.minimum(torch.tensor(img_size[0], device=device).to(torch.int32), torch.round(y_flat + window_size//2 + 1).to(torch.int32)) # (n_vis_points)

#     valid_idxs_x = x_starts < x_ends # If point outside of bounds more than window reaches, no heatmap
#     valid_idxs_y = y_starts < y_ends
#     valid_idxs = valid_idxs_x & valid_idxs_y # Only keep if both x and y within bounds

#     x_starts, y_starts, x_ends, y_ends = x_starts[valid_idxs], y_starts[valid_idxs], x_ends[valid_idxs], y_ends[valid_idxs]
#     corner_types, img_number = corner_types[valid_idxs], img_number[valid_idxs]
#     x_flat, y_flat = x_flat[valid_idxs], y_flat[valid_idxs]

#     windows_X = torch.zeros((len(x_flat), (window_size//2)*2 + 1, (window_size//2)*2 + 1), dtype=torch.float32, device=device)
#     windows_Y = torch.zeros((len(x_flat), (window_size//2)*2 + 1, (window_size//2)*2 + 1), dtype=torch.float32, device=device)

#     for i in range(len(x_flat)):
#         windows_X[i, :y_ends[i]-y_starts[i], :x_ends[i]-x_starts[i]] = X[y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]]
#         windows_Y[i, :y_ends[i]-y_starts[i], :x_ends[i]-x_starts[i]] = Y[y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]]

#     dist_sq = (windows_X - x_flat.unsqueeze(1).unsqueeze(2))**2 + (windows_Y - y_flat.unsqueeze(1).unsqueeze(2))**2
#     gaussian = torch.exp(-dist_sq / (2.0 * sigma**2))

#     for i in range(len(x_flat)):
#         img = int(img_number[i])
#         c = int(corner_types[i])

#         batch_heatmaps[img, c, y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]] = torch.maximum(batch_heatmaps[img, c, y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]], gaussian[i, :y_ends[i]-y_starts[i], :x_ends[i]-x_starts[i]])

#     return batch_heatmaps


# def generate_gt_pafs_vectorized(batch_corners, batch_vis, img_size, device, limb_width=2.0):

#     # batch_corners = (batch_length x n_gates x n_corners(4) x xy(2))
#     # batch_vis = (batch_length x n_gates x n_corners(4))
#     # img_size = (240, 320)

#     batch_length = len(batch_corners)
#     batch_pafs = torch.zeros((batch_length, 8, *img_size), dtype=torch.float32, device=device)
#     limbs = torch.tensor([(0,1), (1,2), (0,3), (3,2)], device=device)  # TL-TR, TR-BR, TL-BL, BL-BR
#     sample_rate = limb_width*2 + 1

#     gates_per_image = torch.tensor([len(corners) for corners in batch_corners], dtype=torch.int32, device=device)
#     mask_no_gates = gates_per_image > 0

#     img_number = torch.arange(batch_length, device=device).repeat_interleave(gates_per_image*4, dim=0).reshape(-1, 4) # [[0,0,0,0],[0,0,0,0],[1,1,1,1],..] --> 2 gates in img 0, 1 gate in img 1

#     corners = torch.concat([batch_corners[i] for i in range(batch_length) if mask_no_gates[i]], dim=0).to(device) # (n_valid_imgs*n_gates x n_corners(4) x xy(2))
#     visibilities = torch.concat([batch_vis[i] for i in range(batch_length) if mask_no_gates[i]], dim=0).to(device) # (n_valid_imgs*n_gates x n_corners(4))
    
#     xy1 = corners[:, limbs[:,0]]  # (n_valid_imgs*n_gates x n_corners(4) x xy(2)) limb start points, basically re-ordered "corners"
#     xy2 = corners[:, limbs[:,1]] # (n_valid_imgs*n_gates x n_corners(4) x xy(2)) limb end points
#     limb_vis = visibilities[:, limbs[:,0]] * visibilities[:, limbs[:,1]]  # (n_valid_imgs*n_gates x n_corners(4)), i.e. limb only visible if both corners visible
#     limb_types = torch.arange(4, device=device).tile(torch.sum(gates_per_image)).reshape(-1, 4) # [[0, 1, 2, 3], [0, 1, 2, 3],..] (total gates_per_image times)

#     mask = limb_vis > 0 # Only select points that are visible
#     if not mask.any():
#         return batch_pafs

#     xy1, xy2, limb_types, img_number = xy1[mask], xy2[mask], limb_types[mask], img_number[mask] # (n_valid_imgs*n_visible_limbs x xy(2)) and (n_valid_imgs*n_visible_limbs)
#     vecs = xy2 - xy1
#     magnitudes = torch.linalg.vector_norm(vecs.to(torch.float32), dim=1) # (n_valid_imgs*n_visible_limbs)
#     line_lengths = torch.max(torch.abs(vecs), dim=1).values + 1 # DDA algorithm

#     valid = line_lengths > 0 # Only vectors longer than 0
#     vecs = vecs[valid] # (n_valid_vecs x xy(2))
#     xy1 = xy1[valid] # (n_valid_vecs x xy(2))
#     limb_types = limb_types[valid] # (n_valid_vecs)
#     img_number = img_number[valid] # (n_valid_vecs)
#     magnitudes = magnitudes[valid] # (n_valid_vecs)
#     line_lengths = line_lengths[valid] # (n_valid_vecs)

#     unit_vecs = vecs / magnitudes.unsqueeze(1) # (n_valid_vecs x xy(2))

#     num_samples = torch.floor((line_lengths-1)/sample_rate).to(torch.int32) + ((line_lengths-1)%sample_rate != 0).to(torch.int32) + 1 # (n_valid_vecs). Only every (limb_width*2 + 1)th sample since the in-between are covered by limb width. Include start and end points.

#     t = torch.cat([torch.linspace(0, 1, int(ns), device=device) for ns in num_samples], dim=0) # (n_valid_vecs*n_samples)
#     points = xy1.repeat_interleave(num_samples, dim=0) + t.unsqueeze(1) * vecs.repeat_interleave(num_samples, dim=0)  # (n_valid_vecs*n_samples x xy(2)) +  (n_valid_vecs*n_samples x 1) * (n_valid_vecs*n_samples x xy(2)) --> (n_valid_vecs*n_samples x xy(2))
#     sample_to_vec = torch.arange(len(num_samples), device=device, dtype=torch.int32).repeat_interleave(num_samples, dim=0) # (n_valid_vecs*n_samples)

#     count = torch.zeros((batch_length, 4, *img_size), device=device) # (n_imgs, 4, 240, 320). To count occurences in same pixel, i.e. limb overlap, per limb type

#     starts = torch.maximum(torch.tensor([0], device=device), torch.round(points - limb_width).to(torch.int32)) # (n_valid_vecs*n_samples x xy(2))
#     ends = torch.minimum(torch.tensor([[img_size[1], img_size[0]]], device=device).to(torch.int32), torch.round(points + limb_width + 1).to(torch.int32)) # (n_valid_vecs*n_samples x xy(2))

#     valid_idxs = starts < ends # (n_valid_vecs*n_samples x 2)
#     valid_idxs = valid_idxs.all(dim=1) # (n_valid_vecs*n_samples)
#     starts, ends = starts[valid_idxs], ends[valid_idxs] # (n_valid_sample_points x xy(2)). Only keep start and end x,y coordinates for those where both x_start < x_end and y_start < y_end
#     sample_to_vec = sample_to_vec[valid_idxs] # (n_valid_sample_points)

#     for i, v in enumerate(sample_to_vec):
#         l = limb_types[v] # Is it limb type 0, 1, 2 or 3?
#         img = img_number[v] # Which image is it?

#         # Add unit vectors, then average by dividing occurences across gates, if there are overlaps
#         batch_pafs[img, 2*l, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += unit_vecs[v, 0] # Within limb, x component of unit-vector
#         batch_pafs[img, 2*l + 1, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += unit_vecs[v, 1] # y component of unit-vector
#         count[img, l, starts[i,1]:ends[i,1], starts[i,0]:ends[i,0]] += 1.0

#     for l in range(4):
#         mask_c = count[:, l] > 1.0 # If a pixel occurred more than once
#         batch_pafs[:, l*2][mask_c] /= count[:, l][mask_c] # Divide paf value by its amount of occurences
#         batch_pafs[:, l*2 + 1][mask_c] /= count[:, l][mask_c]

#     return batch_pafs


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Time elapsed: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# LOSS FUNCTION

start_time = datetime.now()


class WAHRLoss(nn.Module):
    def __init__(self, gamma=0.01, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.mse = nn.MSELoss(reduction='none')

    def forward(self, pred, gt):
        mse_loss = self.mse(pred, gt)
        weight = (gt ** self.gamma) * torch.abs(1 - pred) + (1 - (gt ** self.gamma)) * torch.abs(pred)
        # Bigger weight for positive predictions

        loss = weight * mse_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

def loss_function(batch_pred_heatmaps, batch_gt_heatmaps, batch_pred_pafs, batch_gt_pafs):
    heatmap_loss = WAHRLoss()(batch_pred_heatmaps, batch_gt_heatmaps)
    paf_loss = nn.SmoothL1Loss(reduction='mean')(batch_pred_pafs, batch_gt_pafs)

    return heatmap_loss + paf_loss


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# GET CORNERS FROM PREDICTIONS

start_time = datetime.now()


def find_corner_candidates(heatmaps, threshold=0.3, nms_kernel=5):
    candidates = [[] for _ in range(4)]
    heatmaps = heatmaps.cpu()
    for c in range(4):
        hm = heatmaps[c]

        max_pixels = F.max_pool2d(hm.unsqueeze(0).unsqueeze(0), kernel_size=nms_kernel, stride=1, padding=nms_kernel//2).squeeze()
        peaks_mask = (hm == max_pixels) & (hm > threshold)
        peak_ys, peak_xs = torch.nonzero(peaks_mask, as_tuple=True)
        per_type_cands = []
        
        for py, px in zip(peak_ys, peak_xs): # integers
            heat = hm[py, px].item()
            per_type_cands.append((px.item(), py.item(), heat))
        
        per_type_cands.sort(key=lambda x: x[2], reverse=True) # Sort candidates by heat score
        candidates[c] = per_type_cands[:10] # Take top 10 best candidates per corner
    return candidates


def line_similarity(xy1, xy2, paf_xy, num_samples=10):
    vec = np.array(xy2) - np.array(xy1) # Integer vector

    magnitude = np.sqrt((vec**2).sum())
    if magnitude == 0:
        return 0.0

    unit_vec = vec/magnitude
    t = np.linspace(0, 1, num_samples)
    points = (np.array(xy1) + t.reshape(num_samples, 1) * vec).round().astype(np.int64)

    similarity = 0.0
    num_products = 0
    for px, py in points:
        if 0 <= px < paf_xy.shape[2] and 0 <= py < paf_xy.shape[1]:
            paf_vec = paf_xy[:, py, px].cpu().numpy()

            dot_prod = np.dot(paf_vec, unit_vec)
            similarity += dot_prod
            num_products += 1
    return similarity / num_products if num_products > 0 else 0.0


def find_gates(candidates, pafs, sim_threshold=0.1, is_train=True):
    # candidates = [[(x,y,heat), (x,y,heat),..] for c in range(4)]
    limbs = [(0,1), (1,2), (0,3), (3,2)] # TL-TR, TR-BR, TL-BL, BL-BR
    similarity_scores = [[] for _ in limbs]
    pafs = pafs.cpu()
    for l, (c1_idx, c2_idx) in enumerate(limbs):
        paf_xy = pafs[2*l:2*l+2]  # [2, H, W] tensor
        for c1_cand_idx, c1_cand in enumerate(candidates[c1_idx]):
            for c2_cand_idx, c2_cand in enumerate(candidates[c2_idx]):
                avg_similarity = line_similarity(c1_cand[:2], c2_cand[:2], paf_xy)
                
                if avg_similarity > sim_threshold:
                    weighted_similarity = avg_similarity*(c1_cand[2] + c2_cand[2])/2
                    similarity_scores[l].append((c1_cand_idx, c2_cand_idx, weighted_similarity))

    all_connections = []

    for l, limb_cands in enumerate(similarity_scores):
        num_c1_cands = len(candidates[limbs[l][0]]) # Use all candidates even if some are not chosen, for indexing candidates[c_idx][c_cand_idx] correctly later
        num_c2_cands = len(candidates[limbs[l][1]])

        if not limb_cands:
            continue

        cost_mat = np.zeros((num_c1_cands, num_c2_cands))
        for (c1_cand_idx, c2_cand_idx, sim) in limb_cands:
            cost_mat[c1_cand_idx, c2_cand_idx] = sim
        row_ind, col_ind = linear_sum_assignment(cost_mat, maximize=True)
        # By hungarian bipartite matching you ensure that each corner candidate only appears once (or less) per limb type, so the corner will only be used once for that limb type, thus different gates will not share a corner candidate for the same limb type

        c1_idx, c2_idx = limbs[l]
        matched = [(cost_mat[c1_cand_idx,c2_cand_idx], c1_cand_idx, c2_cand_idx, c1_idx, c2_idx) for c1_cand_idx,c2_cand_idx in zip(row_ind, col_ind)]

        all_connections.extend(matched)

    all_connections.sort(key=lambda x: x[0], reverse=True)
    possible_gates = [] # [{0:c_cand_idx,..}, {...}]
    
    for _, c1_cand_idx, c2_cand_idx, c1_idx, c2_idx in all_connections:
        c1_gate = None # In which possible gate is c1
        c2_gate = None # In which possible gate is c2 (usually same or None)
        
        for i, pg in enumerate(possible_gates): # Check whether any of the 2 corners has already been used (as opposite of start/end of a limb than it is now). If yes, this limb is connected to that limb
            if c1_idx in pg and pg[c1_idx] == c1_cand_idx:
                c1_gate = i
            if c2_idx in pg and pg[c2_idx] == c2_cand_idx:
                c2_gate = i
        
        if c1_gate is None and c2_gate is None: # If none of the c's has been identified
            new_gate = {c1_idx: c1_cand_idx, c2_idx: c2_cand_idx}
            possible_gates.append(new_gate)
            
        elif c1_gate == c2_gate and c1_gate is not None: # If both c's are already part of this gate (through their other limb type), they are already part of the gate thus limb doesn't need to be added
            continue
            
        elif c1_gate is None: # If c2 is part of a gate, this limb should also be part of it
            possible_gates[c2_gate][c1_idx] = c1_cand_idx
            
        elif c2_gate is None: # Same with c1
            possible_gates[c1_gate][c2_idx] = c2_cand_idx
            
        else: # c1_gate != c2_gate, c1_gate != None, c2_gate != None
            if len(possible_gates[c1_gate]) == 2 and len(possible_gates[c2_gate]) == 2: # If c1_gate contains limb1 and c2_gate contains limb2, both part of the same gate but unconnected, merge them
                possible_gates[c1_gate].update(possible_gates[c2_gate])
                del possible_gates[c2_gate]
            # Else: c1 and c2 part of different gates but with stronger similarity since it came earlier, so don't touch
    
    gates = []
    for pg in possible_gates:
        if is_train: # For train, full gates not needed
            gate = [coord for c_idx in range(4) for coord in (candidates[c_idx][pg[c_idx]][:2]+(2.0,) if c_idx in pg.keys() else (0, 0, 0.0))] # gate = [x1, y1, 2.0, x2, y2, 0.0,..]
            gates.extend(gate)
        
        elif len(pg) == 4 and set(pg.keys()) == {0,1,2,3}:
            gate = [coord for c_idx in range(4) for coord in candidates[c_idx][pg[c_idx]][:2]+(2.0,)]  # gate = [x1, y1, 2.0, x2, y2, 2.0,.., x4, y4, 2.0]
            gates.extend(gate)
    
    return gates # gates = [x1, y1, 2.0, x2,.., y4, 2.0, x1, y1, 2.0,..]


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # TEST HEATMAP AND PAF GT GENERATION PT 1

# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=20)
# print(len(train_ds))

# image, corners, visibilities = train_ds[20]
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# heatmaps = generate_gt_heatmaps_vectorized([torch.tensor(corners)], [torch.tensor(visibilities)], (240,320), device=device)[0]
# pafs = generate_gt_pafs_vectorized([torch.tensor(corners)], [torch.tensor(visibilities)], (240,320), device=device)[0]

# img1 = image.numpy().transpose(1,2,0)

# # plt.imshow(img1)
# plt.imshow(pafs.cpu()[7])


In [ ]:
# # TEST HEATMAP AND PAF GT GENERATION PT 2

# pred_gates = find_gates(find_corner_candidates(heatmaps), pafs, is_train=True)
# print(pred_gates)

In [ ]:
# # TEST HEATMAP AND PAF GT GENERATION PT 3



# x_tl,x_tr,x_br,x_bl = [], [], [], []

# y_tl, y_tr, y_br, y_bl = [], [], [] ,[]

# for g in corners: 
    
#     x_tl.append(g[0,0])
#     x_tr.append(g[1,0])
#     x_br.append(g[2,0])
#     x_bl.append(g[3,0])

#     y_tl.append(g[0,1])
#     y_tr.append(g[1,1])
#     y_br.append(g[2,1])
#     y_bl.append(g[3,1])

# plt.subplots(1,2,figsize= (15,15))
# plt.subplot(1,2,1)
# plt.imshow(img1)

# plt.scatter(
#     x_tl,
#     y_tl,
#     c='yellow',
#     marker='o',
#     s=10
# )

# plt.scatter(
#     x_tr,
#     y_tr,
#     c='red',
#     marker='o',
#     s=10
# )

# plt.scatter(
#     x_br,
#     y_br,
#     c='blue',
#     marker='o',
#     s=10
# )

# plt.scatter(
#     x_bl,
#     y_bl,
#     c='green',
#     marker='o',
#     s=10
# )



# plt.subplot(1,2,2)
# plt.imshow(img1)

# colors = ['yellow', 'red', 'blue', 'green']

# for c in range(4):
#     x = pred_gates[c*3::3*4]
#     y = pred_gates[c*3+1::3*4]
    
#     plt.scatter(
#         x,
#         y,
#         c=colors[c],
#         marker='o',
#         s=10
#     )
    
# plt.show()

# pred_visibilities = (np.array(pred_gates[2::3])/2).astype(int).reshape(-1,4)

# print(visibilities)
# print(pred_visibilities)

# # TL = yellow, TR = red, BR = blue, BL = green
# [115, 164, 2.0, 161, 170, 2.0, 0, 0, 0.0, 104, 216, 2.0, 148, 142, 2.0, 130, 195, 2.0, 0, 0, 0.0, 0, 0, 0.0]

In [ ]:
# FIND F1 SCORE ORIGINAL

start_time = datetime.now()


def IoU(gate1, gate2):
    poly1 = Polygon(gate1.reshape(4, 2))
    poly2 = Polygon(gate2.reshape(4, 2))
    if poly2.is_valid == False:
        # invalid polygon
        return 0.0

    intersection_area = poly1.intersection(poly2).area
    union_area = poly1.union(poly2).area

    iou = intersection_area / union_area

    return iou


def match_gates(IoU_matrix):
    num_gt, num_pred = IoU_matrix.shape
    best_match = []
    best_score = 0

    # IoU matrix is always fat (more columns than rows)
    if num_gt > num_pred:
        IoU_matrix = IoU_matrix.T

    # go through all permutations of the gates find the best according to total IoU
    for perm in permutations(range(max(num_gt, num_pred)), min(num_gt, num_pred)):
        current_score = sum(IoU_matrix[i, j] for i, j in enumerate(perm))
        if current_score > best_score:
            best_score = current_score
            best_match = perm

    best_iou_gt = np.zeros(num_gt, dtype=np.float32)
    best_iou_pred = np.zeros(num_pred, dtype=np.float32)
    for i, j in enumerate(best_match):
        gt_idx = j if num_gt > num_pred else i
        pred_idx = i if num_gt > num_pred else j
        best_iou_gt[gt_idx] = IoU_matrix[i, j]
        best_iou_pred[pred_idx] = IoU_matrix[i, j]

    return best_iou_gt, best_iou_pred, best_score


# def score(
#     solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "Id"
# ) -> float:
#     """
#     Calculate the mean F1 score for gate detection predictions based on Intersection over Union (IoU).
#     This function compares the ground truth and predicted gate region.
#     It computes the IoU for each pair of ground truth and predicted boxes, matches them based on the highest IoU,
#     and calculates the F1 score at different IoU thresholds.
#     Args:
#         solution (pd.DataFrame): DataFrame containing the ground truth bounding boxes with a column named "PredictionString".
#         submission (pd.DataFrame): DataFrame containing the predicted bounding boxes with a column named "PredictionString".
#         row_id_column_name (str): The name of the column containing the row identifiers, which will be removed from both DataFrames.
#     Returns:
#         float: The mean F1 score across different IoU thresholds.
#     """
#     del solution[row_id_column_name]
#     del submission[row_id_column_name]

#     gt_string = solution["PredictionString"]
#     pred_string = submission["PredictionString"]

#     iou_dict = {
#         "gt": [],
#         "pred": [],
#     }
#     for gt, pred in zip(gt_string, pred_string):
#         gt = np.array(list(map(float, gt.split(" "))), dtype=np.float32).reshape(-1, 12)
#         if isinstance(pred, float) and isnan(pred):
#             pred = np.empty((0, 12), dtype=np.float32)
#         if isinstance(pred, str):
#             if pred.strip() == "":
#                 pred = np.empty((0, 12), dtype=np.float32)
#             else:
#                 pred = np.array(
#                     list(map(float, pred.split(" "))), dtype=np.float32
#                 ).reshape(-1, 12)

#         # cap num predictions to 10
#         pred = pred[:10]

#         # remove visibility flag
#         gt = gt[:, [0, 1, 3, 4, 6, 7, 9, 10]]
#         pred = pred[:, [0, 1, 3, 4, 6, 7, 9, 10]]

#         # calculate IoU for all pairs
#         pred_IoU = np.zeros((len(gt), len(pred)), dtype=np.float32)
#         for i, gate1 in enumerate(gt):
#             for j, gate2 in enumerate(pred):
#                 iou = IoU(gate1, gate2)
#                 pred_IoU[i, j] = iou

#         # match gates according to highest IoU
#         iou_gt, iou_pred, _ = match_gates(pred_IoU)
#         iou_dict["gt"].append(iou_gt)
#         iou_dict["pred"].append(iou_pred)

#     # compute f1 (precision and recall) at different IoU thresholds
#     thresholds = np.arange(0.5, 1.0, 0.05)
#     f1_score = np.empty_like(thresholds)

#     for i, threshold in enumerate(thresholds):
#         gt_match = np.concatenate([iou > threshold for iou in iou_dict["gt"]])
#         pred_match = np.concatenate([iou > threshold for iou in iou_dict["pred"]])

#         recall = gt_match.sum() / len(gt_match)
#         precision = pred_match.sum() / max(len(pred_match), 1)

#         f1_score[i] = 2 * recall * precision / (recall + precision + 1e-6)

#     return f1_score.mean()


# if __name__ == "__main__":
#     sol = pd.read_csv("solution.csv")
#     sub = pd.read_csv("submission.csv")
#     print(score(sol, sub, "Id"))


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# FIND F1 VALIDATION SCORE

start_time = datetime.now()


def val_score_func(
    solution: list, submission: list, solution_vis: list
) -> float:

    iou_dict = {
        "gt": [],
        "pred": [],
    }
    for i, (gt, pred) in enumerate(zip(solution, submission)):
        pred = np.array([v/320.0 if i%3 == 0 else v/240.0 if i%3 == 1 else v for i,v in enumerate(pred)], dtype=np.float32).reshape(-1, 12)
        gt = gt.numpy().reshape(-1,2) * np.array([1/320.0, 1/240.0], dtype=np.float32) # [[x,y], [x,y],..]
        gt = np.concatenate((gt, solution_vis[i].numpy().reshape(-1,1)), axis=1, dtype=np.float32).flatten().reshape(-1,12)

        keep_rows_mask = np.all(gt[:, 2::3], axis=1) # Remove gates that don't have all 4 corners visible
        gt = gt[keep_rows_mask]

        # cap num predictions to 10
        pred = pred[:10]

        # remove visibility flag
        gt = gt[:, [0, 1, 3, 4, 6, 7, 9, 10]]
        pred = pred[:, [0, 1, 3, 4, 6, 7, 9, 10]]

        # calculate IoU for all pairs
        pred_IoU = np.zeros((len(gt), len(pred)), dtype=np.float32)
        for i, gate1 in enumerate(gt):
            for j, gate2 in enumerate(pred):
                iou = IoU(gate1, gate2)
                pred_IoU[i, j] = iou

        # match gates according to highest IoU
        iou_gt, iou_pred, _ = match_gates(pred_IoU)
        iou_dict["gt"].append(iou_gt)
        iou_dict["pred"].append(iou_pred)

    # compute f1 (precision and recall) at different IoU thresholds
    thresholds = np.arange(0.5, 1.0, 0.05)
    f1_score = np.empty_like(thresholds)

    for i, threshold in enumerate(thresholds):
        gt_match = np.concatenate([iou > threshold for iou in iou_dict["gt"]])
        pred_match = np.concatenate([iou > threshold for iou in iou_dict["pred"]])

        recall = gt_match.sum() / len(gt_match)
        precision = pred_match.sum() / max(len(pred_match), 1)

        f1_score[i] = 2 * recall * precision / (recall + precision + 1e-6)

    return f1_score.mean()


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # TRAINING AND VALIDATION WITH K-FOLD, DISCARDED

# start_time = datetime.now()


# class EarlyStopper:
#     def __init__(self, patience=5, delta=0.001, best_model_path="/kaggle/working/final_model_simple_val.pt"):
#         self.patience = patience
#         self.delta = delta
#         self.counter = 0
#         self.best_score = -float('inf')
#         self.best_epoch = -1
#         self.best_model_path = best_model_path

#     def early_stop(self, current_score, epoch, model, fold):
#         if current_score > self.best_score + self.delta:
#             self.best_score = current_score
#             self.best_epoch = epoch
#             self.counter = 0
            
#             self.best_model_path = f"/kaggle/working/model_fold_{fold}.pt"
#             torch.save(model.state_dict(), self.best_model_path)
#             print(f"Saved model, best Epoch is {self.best_epoch + 1} with a score of {self.best_score}.")
#             # To recover: m = UNet(); m.load_state_dict(torch.load(path)); m.to(device); m.eval()
#             return False

#         self.counter += 1

#         if self.counter >= self.patience:
#             return True

#         return False


# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=20)
# val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=20) # Different dataset for correct transforms

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# kf = GroupKFold(n_splits=5)
# group_ids = train_ds.video_ids

# fold_scores = [] # Best F1 validation score per fold due to early stopper
# fold_epochs = []  # Epoch number at best F1 validation score (starts at 0)
# best_models_paths = []  # Save top 5 paths
# train_losses = [[] for _ in range(5)] # For plotting
# val_f1s = [[] for _ in range(5)] # For plotting, at every 5th epoch
# elapsed_times = []


# for fold, (train_idxs, val_idxs) in enumerate(kf.split(range(len(train_ds)), groups=group_ids)):
#     print(f"Fold {fold+1}/5 started")
#     train_subset = torch.utils.data.Subset(train_ds, train_idxs) # Use the chosen subset of train and val indexes to make a new dataset
#     val_subset = torch.utils.data.Subset(val_ds, val_idxs)

#     train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_batch)
#     val_loader = DataLoader(val_subset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_batch)

#     model = UNet()
#     model.to(device)

#     optimizer = optim.AdamW(model.parameters(), lr=1e-3)
#     scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

#     early_stopper = EarlyStopper()

#     scaler = torch.amp.GradScaler('cuda')

#     for epoch in range(200):
#         model.train()
#         train_loss = 0.0

#         for i, batch in enumerate(train_loader):    
#             batch_images, batch_corners, batch_visibilities = batch
#             batch_images = batch_images.to(device)
#             batch_size = len(batch_visibilities)

#             gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, batch_size, device)
#             gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, batch_size, device)

#             optimizer.zero_grad()

#             pred_heatmaps, pred_pafs = model(batch_images)
#             loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)
            
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()

#             train_loss += loss.item()

#             print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Avg Train Loss: {train_loss / (i+1)}")

#         train_losses[fold].append(train_loss / len(train_loader))
#         print(f"Fold {fold+1}/5, Epoch {epoch+1}/200, Avg Train Loss: {train_loss / len(train_loader)}")

#         if epoch % 5 == 4: # Validate every 5 epochs
#             val_preds = [] # [[x1, y1, 2.0, x2, y2,..], [x1, y1,..],..]
#             val_gt_gates = [] # [[[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..],..    ],..]
#             val_gt_visibilities = []
#             model.eval()

#             with torch.no_grad():
#                 for batch in val_loader:
#                     batch_images, batch_corners, batch_visibilities = batch
#                     batch_images = batch_images.to(device)
#                     pred_heatmaps, pred_pafs = model(batch_images)

#                     for i in range(len(batch_images)):
#                         corner_candidates = find_corner_candidates(pred_heatmaps[i])
#                         pred_gates = find_gates(corner_candidates, pred_pafs[i], is_train=False)

#                         val_preds.append(pred_gates)
#                         val_gt_gates.append(batch_corners[i])
#                         val_gt_visibilities.append(batch_visibilities[i])

#             val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
#             val_f1s[fold].append(val_score)

#             print(f"Fold {fold+1}/5, Epoch {epoch+1}/200, Val F1 score: {val_score}, LR: {scheduler.get_last_lr()}")

#             scheduler.step(val_score)

#             if early_stopper.early_stop(val_score, epoch, model):
#                 print(f"Fold {fold+1}/5 early stopped at epoch {epoch+1}, best epoch: {early_stopper.best_epoch}")
#                 break

#     # Save info
#     fold_scores.append(early_stopper.best_score)
#     fold_epochs.append(early_stopper.best_epoch)
#     model_path = f"/kaggle/working/model_fold_{fold}.pt"
#     best_models_paths.append(model_path)

    
#     end_time_val = datetime.now()
#     time_elapsed_val = end_time_val - start_time
#     elapsed_times.append(str(time_elapsed_val))
#     print(f"Time for K-fold {fold} since start: {time_elapsed_val}")


# # Keep top 5 models
# top_indices = np.argsort(fold_scores)[::-1]  # Get indices of models ordered by score
# top_paths = [best_models_paths[i] for i in top_indices]
# top_epochs = [fold_epochs[i] for i in top_indices]
# print(f"Top 5 fold paths: {top_paths}")

# valid_results = {
#     "fold_number": top_indices,
#     "top_scores": fold_scores.sort(reverse=True),
#     "top_epochs": top_epochs,
#     "top_model_paths": top_paths,
#     "train_losses_per_epoch": train_losses,
#     "val_f1s_per_5th_epoch": val_f1s,
#     "validation_times_since_start": elapsed_times
# }

# with open("/kaggle/working/validation_results.json", "w") as f:
#     json.dump(valid_results, f, indent=4)
# print("Validation results stored")
# # To recover: json.load(f)

# # Avg best epochs
# best_epochs = int(np.mean(top_epochs))
# print(f"Avg best epochs: {best_epochs}")


# start_time_train = datetime.now()
# scaler = torch.amp.GradScaler('cuda')


# # Re-train
# full_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4, collate_fn=collate_batch)
# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3)
# scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

# final_train_losses = []

# for epoch in range(100): # Or best_epochs + 1
#     print(f"Starting epoch {epoch+1}/100")
#     model.train()
#     train_loss = 0.0

#     for i,batch in enumerate(full_loader): 
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)
#         batch_size = len(batch_visibilities)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         optimizer.zero_grad()

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         train_loss += loss.item()
        
#         time_batch = datetime.now()
#         print(f"Finished batch number {i+1}/{len(full_loader)}, Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_train}")

#     final_train_losses.append(train_loss / len(full_loader))
#     print(f"Full Train Epoch {epoch+1}/100, Avg Train Loss: {train_loss / len(full_loader)}")

#     torch.save(model.state_dict(), "/kaggle/working/final_model.pt")
#     train_results = {
#         "model_path": "/kaggle/working/final_model.pt",
#         "train_losses_per_epoch": final_train_losses,
#         "time_elapsed_train": str(datetime.now()-start_time_train)
#     }
    
#     with open("/kaggle/working/train_results.json", "w") as f:
#         json.dump(train_results, f, indent=4)
#     print("Train results stored")


# end_time_train = datetime.now()
# time_elapsed_train = end_time_train - start_time_train
# print(f"Time elapsed for training: {time_elapsed_train}")


# train_results = {
#     "model_path": "/kaggle/working/final_model.pt",
#     "train_losses_per_epoch": final_train_losses,
#     "time_elapsed_train": str(time_elapsed_train)
# }

# with open("/kaggle/working/train_results.json", "w") as f:
#     json.dump(train_results, f, indent=4)
# print("All train results stored")

# print("Training completed!")


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Time elapsed for full validation and training: {time_elapsed}")
# print(f"Total time: {total_time}")


# # Training times: 
# # Old version, 50/495 batches, batch size 16, non-vectorized, 187.090474
# # Frame rate 20:
# # 100/990 batches, batch size 8, vectorized, 103.936183. 2nd try: 105.857762 (200 batches 211.771808, /2 = 105.885904) --> 1 epoch = 17:28 min
# # 50/495 batches, batch size 16, vectorized, 106.674084. 2nd try: 102.302779 (200 batches 205.060813, /2 = 102.5304065) --> 1 epoch = 16:55 min
# # 50/495 batches, batch size 16, non-vectorized, 97.047515 (100 batches 194.828691, /2 = 97.4143) --> 1 epoch = 16:04 min
# # 100/990 batches, batch size 8, non-vectorized, 100.488205 (200 batches 202.255757, /2 = 101.1278785) --> 1 epoch = 16:41 min
# # 25/248 batches, batch size 32, vectorized, 100.093411 (50 batches 200.37641, /2 = 100.188205) --> 1 epoch = 16:34 min
# # 25/248 batches, batch size 32, non-vectorized, 96.988917 (50 batches 193.104077, /2 = 96.5520385) --> 1 epoch = 15:58 min or 12:48 min for 80% of data (6:24 for frame rate 40)

# # Test times:
# # Frame rate 20:
# # 50/495 batches, batch size 16, 94.798 (100 batches 186.508877, /2 = 93.2544) --> 1 loop through 20% of data = 3:05 min
# # 100/990 batches, batch size 8, 89.206672 (200 batches 175.150122, /2 = 87.575061) --> 1 loop through 20% of data = 2:53 min (1:27 for frame rate 40)
# # 800/7917 batches, batch size 1, 89.934446

# # 50 epochs with frame rate 40: 320 (train) + 14:30 (val) = 5h, 35 min (6h, 33 min validating at every epoch). Training: 6h 40 min

In [ ]:
# # SIMPLE VALIDATION ON FULL DATASET, SUBMISSION V1

# start_time = datetime.now()


# class EarlyStopper:
#     def __init__(self, patience=5, delta=0.001, best_model_path="/kaggle/working/final_model_simple_val.pt"):
#         self.patience = patience
#         self.delta = delta
#         self.counter = 0
#         self.best_score = -float('inf')
#         self.best_epoch = -1
#         self.best_model_path = best_model_path

#     def early_stop(self, current_score, epoch, model):
#         if current_score > self.best_score + self.delta:
#             self.best_score = current_score
#             self.best_epoch = epoch
#             self.counter = 0
#             torch.save(model.state_dict(), self.best_model_path)
#             print(f"Saved model, best Epoch is {self.best_epoch + 1} with a score of {self.best_score}.")
#             # To recover: m = UNet(); m.load_state_dict(torch.load(path)); m.to(device); m.eval()
#             return False

#         self.counter += 1

#         if self.counter >= self.patience:
#             return True

#         return False


# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

# batch_size = 32
# frame_rate = 40

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)
# val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=frame_rate) # Different dataset for correct transforms

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# rs = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
# train_idxs, val_idxs = next(rs.split(range(len(train_ds)))) # Split dataset indexes into train and val

# train_subset = torch.utils.data.Subset(train_ds, train_idxs) # Use the chosen subset of train and val indexes to make a new dataset
# val_subset = torch.utils.data.Subset(val_ds, val_idxs)

# train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=collate_batch)
# val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_batch)

# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Starting learning rate 1e-3
# scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True) # Reduce learning rate to half (factor), if validation score didn't increase in the last 3 times (patience) you did a validation, i.e. could be every 5*3 epochs. In case of not reaching minima.
# early_stopper = EarlyStopper(patience=11)
# scaler = torch.amp.GradScaler('cuda')

# validate_every_nth_epoch = 1

# train_losses = [] # For plotting
# val_f1s = [] # For plotting, at every 5th epoch

# for epoch in range(100):
#     start_time_epoch = datetime.now()
#     print(f"Starting Epoch {epoch+1}/100.")
#     model.train()
#     train_loss = 0.0

#     for batch in train_loader:    
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         train_loss += loss.item()
        
#         # time_batch = datetime.now()
#         # print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Train Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_epoch}")
    
#     time_train_epoch = datetime.now()
#     train_losses.append(train_loss / len(train_loader))
#     print(f"Trained Epoch {epoch+1}/100, Avg Train Loss for this Epoch: {train_loss / len(train_loader)}, it took: {time_train_epoch-start_time_epoch}")

#     # last_trained_model = model.state_dict()
#     # torch.save(last_trained_model, f"/kaggle/working/last_trained_model_epoch_{epoch+1}.pt")

#     if epoch > 8:
#         torch.save(model.state_dict(), f"/kaggle/working/model_epoch_{epoch+1}.pt")
#         print(f"Saved model for Epoch {epoch+1}, it took {datetime.now() - time_train_epoch}")

#     if epoch > 8 and (epoch+1) % validate_every_nth_epoch == 0: # Validate every nth epoch starting from epoch 10
#         start_time_epoch_val = datetime.now()
#         val_preds = [] # [[x1, y1, 2.0, x2, y2,..], [x1, y1,..],..]
#         val_gt_gates = [] # [[[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..],..    ],..]
#         val_gt_visibilities = []
#         model.eval()

#         with torch.no_grad():
#             for batch in val_loader:
#                 start_time_batch_val = datetime.now()
#                 # print(f"Started Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, at {start_time_batch_val-start_time_epoch_val}")
#                 batch_images, batch_corners, batch_visibilities = batch
#                 batch_images = batch_images.to(device)
#                 pred_heatmaps, pred_pafs = model(batch_images)
                
#                 for i in range(len(batch_images)):
#                     corner_candidates = find_corner_candidates(pred_heatmaps[i])
#                     pred_gates = find_gates(corner_candidates, pred_pafs[i], is_train=False)

#                     val_preds.append(pred_gates)
#                     val_gt_gates.append(batch_corners[i])
#                     val_gt_visibilities.append(batch_visibilities[i])

#                 # print(f"Finished Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, after {datetime.now() - start_time_batch_val}")
                
                    
#         print(f"Validation Epoch completed, now scoring function. It took {datetime.now() - start_time_epoch_val}")

#         val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
#         val_f1s.append(val_score)

#         end_time_epoch_val = datetime.now()
#         print(f"Validated Epoch {epoch+1}/100, Val F1 score: {val_score}, LR: {scheduler.get_last_lr()}, it took: {end_time_epoch_val - start_time_epoch_val}")

#         scheduler.step(val_score)

#         if early_stopper.early_stop(val_score, epoch, model):
#             print(f"Early stopped at epoch {epoch+1}, best epoch: {early_stopper.best_epoch + 1}, best score: {early_stopper.best_score}")
#             break

# # Save info
# best_score = early_stopper.best_score # Best F1 validation score due to early stopper
# best_epoch = early_stopper.best_epoch # Epoch number at best F1 validation score (starts at 0)
# best_model_path = f"/kaggle/working/final_model_simple_val.pt"
# print(f"Best model path: {best_model_path}, best epoch: {best_epoch}, best score: {best_score}")
# print(f"For Plotting. Train losses per epoch: {train_losses}, validation F1 every {validate_every_nth_epoch}th epoch, starting at 10: {val_f1s}")

# end_time_val = datetime.now()
# time_elapsed_val = end_time_val - start_time
# print(f"Time for full validation since start: {time_elapsed_val}")

# valid_results = {
#     "best_score": str(best_score),
#     "best_epoch": str(best_epoch),
#     "best_model_path": best_model_path,
#     "train_losses_per_epoch": train_losses,
#     f"val_f1s_per_{validate_every_nth_epoch}th_epoch": val_f1s,
#     "validation_time_since_start": str(time_elapsed_val)
# }

# with open("/kaggle/working/validation_results_1.json", "w") as f:
#     json.dump(valid_results, f, indent=4)
# print("Validation results stored")
# # To recover: json.load(f)


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Final time elapsed for validation: {time_elapsed}")
# print(f"Total time: {total_time}")


# # Training times: 
# # Old version, 50/495 batches, batch size 16, non-vectorized, 187.090474
# # Frame rate 20:
# # 100/990 batches, batch size 8, vectorized, 103.936183. 2nd try: 105.857762 (200 batches 211.771808, /2 = 105.885904) --> 1 epoch = 17:28 min
# # 50/495 batches, batch size 16, vectorized, 106.674084. 2nd try: 102.302779 (200 batches 205.060813, /2 = 102.5304065) --> 1 epoch = 16:55 min
# # 50/495 batches, batch size 16, non-vectorized, 97.047515 (100 batches 194.828691, /2 = 97.4143) --> 1 epoch = 16:04 min
# # 100/990 batches, batch size 8, non-vectorized, 100.488205 (200 batches 202.255757, /2 = 101.1278785) --> 1 epoch = 16:41 min
# # 25/248 batches, batch size 32, vectorized, 100.093411 (50 batches 200.37641, /2 = 100.188205) --> 1 epoch = 16:34 min
# # 25/248 batches, batch size 32, non-vectorized, 96.988917 (50 batches 193.104077, /2 = 96.5520385) --> 1 epoch = 15:58 min or 12:48 min for 80% of data (6:24 for frame rate 40)

# # Test times:
# # Frame rate 20:
# # 50/495 batches, batch size 16, 94.798 (100 batches 186.508877, /2 = 93.2544) --> 1 loop through 20% of data = 3:05 min
# # 100/990 batches, batch size 8, 89.206672 (200 batches 175.150122, /2 = 87.575061) --> 1 loop through 20% of data = 2:53 min (1:27 for frame rate 40)
# # 800/7917 batches, batch size 1, 89.934446

# # 50 epochs with frame rate 40: 320 (train) + 14:30 (val) = 5h, 35 min (6h, 33 min validating at every epoch). Training: 6h 40 min

In [ ]:
# # PLOT VALIDATION PROGRESS FOR SUBMISSION V1

# train_losses = 100*np.array([0.025797678495950577, 0.0036015384369266204, 0.002918164485204473, 0.0027129678429051673, 0.0025892316952418042, 0.002460219809469187, 0.0020974540922632616, 0.0017163094609782522, 0.0014520134746079457, 0.0012829666528938655, 0.0011467856941481103, 0.0010199053081209367, 0.0009658107257289988, 0.0009095407017497873, 0.000856572143812576, 0.0008198487520753846, 0.0007628925295133706, 0.0007483368796595143, 0.0007309863280135927, 0.0007005603907120256, 0.0006891784559109441, 0.0006677568571659613, 0.0006559834316403623, 0.0006425206649407112, 0.0006302293905667644, 0.0006212748374069497, 0.0006065323516695742, 0.0005966797347492451, 0.0005838005945749827, 0.0005696402768766999, 0.000542972175918414, 0.0005189620157734509, 0.0005235124685810352, 0.0005148931566783982, 0.0005186195114631873, 0.0005144672756597294, 0.0004985610293312315, 0.0005010661420869237, 0.0005003134582745228, 0.0004781171029328175, 0.0004544641700669912, 0.00046472572106516584, 0.00045327010168035004, 0.00046783139191146166, 0.00045560728066671146, 0.0004530172325002978, 0.0004434582638940862, 0.000444092353441278, 0.00044285672203910027, 0.0004486027895802421, 0.00044006112148053944, 0.00044686307571608713, 0.0004380905250074201, 0.00043573981301446075, 0.00042410548128416094, 0.0004221575671284161, 0.0004178120621220859, 0.00041978760404986733, 0.0004146132271688537, 0.00042201471152574806, 0.0004215879206005308, 0.00040958193048421855, 0.00040945923175522177, 0.00040888486766115813, 0.0004085140140235143, 0.000399487180127938, 0.0004085866080372998, 0.0004036100117274437, 0.0003881649680343404, 0.0003848596182783609, 0.0003849970567246618, 0.00038979211775476186])
# val_f1s = [0.7291648683711401, 0.8008492741699982, 0.820142103196479, 0.8133290030216844, 0.8493394826962275, 0.8579450665273974, 0.8771240635789697, 0.8676508177588367, 0.8779208321885272, 0.8872427745670134, 0.8791855466261704, 0.8961822646524368, 0.8918179446829726, 0.9066545926471253, 0.914557558963299, 0.9046034462409358, 0.9002015202326609, 0.9041059199445114, 0.9120365370915685, 0.8997399096022839, 0.9058818529434743, 0.9132789457604286, 0.9133866898726147, 0.9164697500791184, 0.9128787215230515, 0.9133730979388321, 0.913340500583422, 0.9121736621315139, 0.9126464744779341, 0.9138748231882634, 0.9214260171220336, 0.9138756467899262, 0.9176803266389622, 0.9073642871224952, 0.9216156616467954, 0.9133212993321825, 0.9232934821632377, 0.9144739399814256, 0.928739770993713, 0.9227712078864665, 0.9238556151201536, 0.928015624403374, 0.9154770604251393, 0.9250499468368962, 0.9231383330052436, 0.9250209961313391, 0.9263395576572583, 0.92588928186284, 0.9288423324834701, 0.930114442534952, 0.9232353003459884, 0.9325850622758975, 0.9285010818329884, 0.9268119280891165, 0.9315299948324988, 0.9284846969029132, 0.9250067108713822, 0.9277692841849003, 0.9262545490763017, 0.9268175511713814, 0.9280409150225193, 0.9222089361443542, 0.9285052471326537]

# epoch_plotting = list(range(1, len(train_losses)+1))
# validate_every_nth_epoch = 1

# plt.plot(epoch_plotting, train_losses, label='Train loss')
# # plt.yscale("log")
# plt.plot(epoch_plotting[9::validate_every_nth_epoch], val_f1s, label='Validation F1')
# plt.scatter(epoch_plotting[59], val_f1s[59-10], label="Best epoch")
# plt.legend()
# plt.show()

In [ ]:
# # BEST PRIVATE SCORE MODEL (V1 FINAL EPOCH)

# start_time = datetime.now()

# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# batch_size = 32
# frame_rate = 40

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# rs = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
# train_idxs, _ = next(rs.split(range(len(train_ds)))) # Split dataset indexes into train and val

# train_subset = torch.utils.data.Subset(train_ds, train_idxs) # Use the chosen subset of train indexes to make a new dataset

# train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=collate_batch)

# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Starting learning rate 1e-3
# scaler = torch.amp.GradScaler('cuda')

# train_losses = [] # For plotting

# LRs_best = [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 0.000125, 6.25e-05, 6.25e-05, 6.25e-05, 6.25e-05, 6.25e-05]

# for epoch in range(72):
#     start_time_epoch = datetime.now()
#     print(f"Starting Epoch {epoch+1}/72.")
#     model.train()
#     train_loss = 0.0

#     for param_group in optimizer.param_groups:
#         param_group["lr"] = LRs_best[epoch]

#     for batch in train_loader:    
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         train_loss += loss.item()
        
#         # time_batch = datetime.now()
#         # print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Train Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_epoch}")
    
#     time_train_epoch = datetime.now()
#     train_losses.append(train_loss / len(train_loader))
#     print(f"Trained Epoch {epoch+1}/72, Avg Train Loss for this Epoch: {train_loss / len(train_loader)}, it took: {time_train_epoch-start_time_epoch}")

# # # Save info
# # best_model_path = f"/kaggle/working/final_model_simple_val.pt"
# # print(f"Best model path: {best_model_path}")
# # print(f"For Plotting. Train losses per epoch: {train_losses}")

# # valid_results = {
# #     "best_model_path": best_model_path,
# #     "train_losses_per_epoch": train_losses,
# # }

# # with open("/kaggle/working/validation_results_1.json", "w") as f:
# #     json.dump(valid_results, f, indent=4)
# # print("Validation results stored")
# # # To recover: json.load(f)


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Final time elapsed for validation: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# # SIMPLE TRAINING ON FULL DATASET, SUBMISSION V1

# start_time_train = datetime.now()


# batch_size = 32
# frame_rate = 70

# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")
# scaler = torch.amp.GradScaler('cuda')

# full_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=collate_batch)
# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# # final_train_losses = []

# for epoch in range(72): 
#     start_time_train_epoch = datetime.now()
#     print(f"Starting epoch {epoch+1}/72, Time: {start_time_train_epoch - start_time_train}")
#     model.train()
#     train_loss = 0.0

#     for batch in full_loader: 
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         train_loss += loss.item()
        
#         # time_batch = datetime.now()
#         # print(f"Finished batch number {i+1}/{len(full_loader)}, Loss: {train_loss/(i+1)}, it took: {time_batch-start_time_train_epoch}")

#     # final_train_losses.append(train_loss / len(full_loader))
#     print(f"Fully trained epoch {epoch+1}/72, Avg Train Loss: {train_loss / len(full_loader)}")

#     torch.save(model.state_dict(), "/kaggle/working/current_final_model_full_dataset.pt")
#     # train_results = {
#     #     "model_path": "/kaggle/working/final_model.pt",
#     #     "train_losses_per_epoch": final_train_losses,
#     #     "time_elapsed_train": str(datetime.now()-start_time_train)
#     # }
    
#     # with open("/kaggle/working/train_results.json", "w") as f:
#     #     json.dump(train_results, f, indent=4)
#     # print("Train results stored")


# end_time_train = datetime.now()
# time_elapsed_train = end_time_train - start_time_train
# final_model_path = "/kaggle/working/final_model_full_dataset.pt"
# torch.save(model.state_dict(), final_model_path)

# print(f"Time elapsed for training: {time_elapsed_train}")
# print(f"Path for final model: {final_model_path}")


# train_results = {
#     "model_path": final_model_path,
#     "time_elapsed_train": str(time_elapsed_train)
# }

# with open("/kaggle/working/train_results_1.json", "w") as f:
#     json.dump(train_results, f, indent=4)
# print("All train results stored")

# print("Training completed!")


# end_time = datetime.now()
# time_elapsed = end_time - start_time_train
# total_time += time_elapsed

# print(f"Time elapsed for training: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# # SIMPLE VALIDATION USING NON-VIDEOS AS VAL, SUBMISSION V3

# start_time = datetime.now()


# class EarlyStopper:
#     def __init__(self, patience=5, delta=0.001, best_model_path="/kaggle/working/final_model_simple_val_no_videos.pt"):
#         self.patience = patience
#         self.delta = delta
#         self.counter = 0
#         self.best_score = -float('inf')
#         self.best_epoch = -1
#         self.best_model_path = best_model_path

#     def early_stop(self, current_score, epoch, model):
#         if current_score > self.best_score + self.delta:
#             self.best_score = current_score
#             self.best_epoch = epoch
#             self.counter = 0
#             torch.save(model.state_dict(), self.best_model_path)
#             print(f"Saved model, best Epoch is {self.best_epoch + 1} with a score of {self.best_score}.")
#             # To recover: m = UNet(); m.load_state_dict(torch.load(path)); m.to(device); m.eval()
#             return False

#         self.counter += 1

#         if self.counter >= self.patience:
#             return True

#         return False


# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

# batch_size = 32
# frame_rate = 40

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)
# val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=frame_rate) # Different dataset for correct transforms

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# val_length = 30 + 56 + 141 + 195 + 41 + 36 # The non-video files

# val_idxs = [i for i in range(val_length)] # Split dataset indexes into train and val, using non-video for val
# train_idxs = [i for i in range(val_length, len(train_ds))] 

# train_subset = torch.utils.data.Subset(train_ds, train_idxs) # Use the chosen subset of train and val indexes to make a new dataset
# val_subset = torch.utils.data.Subset(val_ds, val_idxs)

# train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=collate_batch)
# val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_batch)

# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Starting learning rate 1e-3
# scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True) # Reduce learning rate to half (factor), if validation score didn't increase in the last 3 times (patience) you did a validation, i.e. could be every 5*3 epochs. In case of not reaching minima.
# early_stopper = EarlyStopper(patience=11)
# scaler = torch.amp.GradScaler('cuda')

# validate_every_nth_epoch = 1

# train_losses = [] # For plotting
# val_f1s = [] # For plotting, at every 5th epoch
# LRs = [] # Save the learning rate for each epoch

# for epoch in range(100):
#     start_time_epoch = datetime.now()
#     print(f"Starting Epoch {epoch+1}/100.")
#     model.train()
#     train_loss = 0.0

#     p_groups = len(optimizer.param_groups)
#     if p_groups == 1:
#         LRs.append(optimizer.param_groups[0]["lr"]) # Save LRs
#     else:
#         print("More than 1 param group in optimizer.")
#         set_lrs = set([optimizer.param_groups[p_group]["lr"] for p_group in range(p_groups)])

#         if len(set_lrs) == 1:
#             LRs.append(optimizer.param_groups[0]["lr"])
#         else:
#             LRs.append(list(set_lrs))
        
#     for batch in train_loader:    
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         train_loss += loss.item()
        
#         # time_batch = datetime.now()
#         # print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Train Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_epoch}")
    
#     time_train_epoch = datetime.now()
#     train_losses.append(train_loss / len(train_loader))
#     print(f"Trained Epoch {epoch+1}/100, Avg Train Loss for this Epoch: {train_loss / len(train_loader)}, it took: {time_train_epoch-start_time_epoch}")
     
#     if epoch > 8 and (epoch+1) % validate_every_nth_epoch == 0: # Validate every nth epoch starting from epoch 10
#         print(f"LRs so far at Epoch {epoch+1}: {LRs}")
        
#         start_time_epoch_val = datetime.now()
#         val_preds = [] # [[x1, y1, 2.0, x2, y2,..], [x1, y1,..],..]
#         val_gt_gates = [] # [[[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..],..    ],..]
#         val_gt_visibilities = []
#         model.eval()

#         with torch.no_grad():
#             for batch in val_loader:
#                 start_time_batch_val = datetime.now()
#                 # print(f"Started Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, at {start_time_batch_val-start_time_epoch_val}")
#                 batch_images, batch_corners, batch_visibilities = batch
#                 batch_images = batch_images.to(device)
#                 pred_heatmaps, pred_pafs = model(batch_images)
                
#                 for i in range(len(batch_images)):
#                     corner_candidates = find_corner_candidates(pred_heatmaps[i])
#                     pred_gates = find_gates(corner_candidates, pred_pafs[i], is_train=False)

#                     val_preds.append(pred_gates)
#                     val_gt_gates.append(batch_corners[i])
#                     val_gt_visibilities.append(batch_visibilities[i])

#                 # print(f"Finished Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, after {datetime.now() - start_time_batch_val}")
                
                    
#         print(f"Validation Epoch completed, now scoring function. It took {datetime.now() - start_time_epoch_val}")

#         val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
#         val_f1s.append(val_score)

#         end_time_epoch_val = datetime.now()
#         print(f"Validated Epoch {epoch+1}/100, Val F1 score: {val_score}, LR: {scheduler.get_last_lr()}, it took: {end_time_epoch_val - start_time_epoch_val}")

#         scheduler.step(val_score)

#         if early_stopper.early_stop(val_score, epoch, model):
#             print(f"Early stopped at epoch {epoch+1}, best epoch: {early_stopper.best_epoch + 1}, best score: {early_stopper.best_score}")
#             break

# # Save info
# best_score = early_stopper.best_score # Best F1 validation score due to early stopper
# best_epoch = early_stopper.best_epoch # Epoch number at best F1 validation score (starts at 0)
# best_model_path = f"/kaggle/working/final_model_simple_val_no_videos.pt"
# print(f"Best model path: {best_model_path}, best epoch: {best_epoch}, best score: {best_score}, LRs: {LRs}")
# print(f"For Plotting. Train losses per epoch: {train_losses}, validation F1 every {validate_every_nth_epoch}th epoch, starting at 10: {val_f1s}")

# end_time_val = datetime.now()
# time_elapsed_val = end_time_val - start_time
# print(f"Time for full validation since start: {time_elapsed_val}")

# valid_results = {
#     "best_score": str(best_score),
#     "best_epoch": str(best_epoch),
#     "LRs": LRs,
#     "best_model_path": best_model_path,
#     "train_losses_per_epoch": train_losses,
#     f"val_f1s_per_{validate_every_nth_epoch}th_epoch": val_f1s,
#     "validation_time_since_start": str(time_elapsed_val)
# }

# with open("/kaggle/working/validation_results_2.json", "w") as f:
#     json.dump(valid_results, f, indent=4)
# print("Validation results stored")
# # To recover: json.load(f)


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Final time elapsed for validation: {time_elapsed}")
# print(f"Total time: {total_time}")


# # Training times: 
# # Old version, 50/495 batches, batch size 16, non-vectorized, 187.090474
# # Frame rate 20:
# # 100/990 batches, batch size 8, vectorized, 103.936183. 2nd try: 105.857762 (200 batches 211.771808, /2 = 105.885904) --> 1 epoch = 17:28 min
# # 50/495 batches, batch size 16, vectorized, 106.674084. 2nd try: 102.302779 (200 batches 205.060813, /2 = 102.5304065) --> 1 epoch = 16:55 min
# # 50/495 batches, batch size 16, non-vectorized, 97.047515 (100 batches 194.828691, /2 = 97.4143) --> 1 epoch = 16:04 min
# # 100/990 batches, batch size 8, non-vectorized, 100.488205 (200 batches 202.255757, /2 = 101.1278785) --> 1 epoch = 16:41 min
# # 25/248 batches, batch size 32, vectorized, 100.093411 (50 batches 200.37641, /2 = 100.188205) --> 1 epoch = 16:34 min
# # 25/248 batches, batch size 32, non-vectorized, 96.988917 (50 batches 193.104077, /2 = 96.5520385) --> 1 epoch = 15:58 min or 12:48 min for 80% of data (6:24 for frame rate 40)

# # Test times:
# # Frame rate 20:
# # 50/495 batches, batch size 16, 94.798 (100 batches 186.508877, /2 = 93.2544) --> 1 loop through 20% of data = 3:05 min
# # 100/990 batches, batch size 8, 89.206672 (200 batches 175.150122, /2 = 87.575061) --> 1 loop through 20% of data = 2:53 min (1:27 for frame rate 40)
# # 800/7917 batches, batch size 1, 89.934446

# # 50 epochs with frame rate 40: 320 (train) + 14:30 (val) = 5h, 35 min (6h, 33 min validating at every epoch). Training: 6h 40 min

In [ ]:
# SIMPLE (ENHANCED) VALIDATION, OVER-SAMPLE NON-VIDEOS, SUBMISSION V2

start_time = datetime.now()


class EarlyStopper:
    def __init__(self, patience=5, delta=0.001, best_model_path="/kaggle/working/final_model_enhanced_validation.pt"):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = -float('inf')
        self.best_epoch = -1
        self.best_model_path = best_model_path

    def early_stop(self, current_score, epoch, model):
        if current_score > self.best_score + self.delta:
            self.best_score = current_score
            self.best_epoch = epoch
            self.counter = 0
            torch.save(model.state_dict(), self.best_model_path)
            print(f"Saved model, best Epoch is {self.best_epoch + 1} with a score of {self.best_score}.")
            # To recover: m = UNet(); m.load_state_dict(torch.load(path)); m.to(device); m.eval()
            return False

        self.counter += 1

        if self.counter >= self.patience:
            return True

        return False


train_transforms = Compose([
    Rotate(limit=30, p=0.5),
    RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    GaussNoise(std_range=(0.2,0.3), p=0.3),
    Blur(blur_limit=3, p=0.2),
    Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
    ToTensorV2()
], keypoint_params={'format': 'xy', 'remove_invisible': False})

val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

batch_size = 32
frame_rate = 70 # About 2000 video samples

train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)
val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=frame_rate) # Different dataset for correct transforms

compensated_indexes = np.array([i for i in range(train_ds.cum_sizes[5]) for _ in range(4)] + [i for i in range(train_ds.cum_sizes[5], len(train_ds))]) # Take non-video images 4 times, rest once
updated_file_lengths = [(file_lengths[i]-1)//frame_rate + 1 if f.startswith(('autonomous', 'piloted')) else file_lengths[i]*4 for (i,f) in enumerate(filenames)]
updated_video_ids = np.array([file_idx for file_idx, length in enumerate(updated_file_lengths) for _ in range(length)])

current_train_size = 0
goal_train_size = round(0.8*sum(updated_file_lengths))
train_groups, val_groups = [], []

np.random.seed(42)
for g in np.random.choice(len(updated_file_lengths), len(updated_file_lengths), replace=False): # Choose random indexes
    if current_train_size + updated_file_lengths[g] <= goal_train_size:
        train_groups.append(g)
        current_train_size += updated_file_lengths[g]
    else:
        val_groups.append(g)

train_ids = np.concatenate([np.where(updated_video_ids == g)[0] for g in train_groups])
val_ids = np.concatenate([np.where(updated_video_ids == g)[0] for g in val_groups])

train_idxs = compensated_indexes[train_ids]
val_idxs = compensated_indexes[val_ids]

print(f"{len(train_idxs)} train files, {len(val_idxs)} validation files")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used: {device}")

torch.manual_seed(42)
train_sampler = torch.utils.data.SubsetRandomSampler(train_idxs)
val_sampler = torch.utils.data.SubsetRandomSampler(val_idxs)

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, num_workers=4, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=batch_size, sampler=val_sampler, num_workers=4, collate_fn=collate_batch)

model = UNet()
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Starting learning rate 1e-3
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True) # Reduce learning rate to half (factor), if validation score didn't increase in the last 3 times (patience) you did a validation, i.e. could be every 5*3 epochs. In case of not reaching minima.
early_stopper = EarlyStopper(patience=11)
scaler = torch.amp.GradScaler('cuda')

validate_every_nth_epoch = 1

train_losses = [] # For plotting
val_f1s = [] # For plotting, at every 5th epoch
LRs = [] # Save the learning rate for each epoch

for epoch in range(100):
    start_time_epoch = datetime.now()
    print(f"Starting Epoch {epoch+1}/100.")
    model.train()
    train_loss = 0.0

    p_groups = len(optimizer.param_groups)
    if p_groups == 1:
        LRs.append(optimizer.param_groups[0]["lr"]) # Save LRs
    else:
        print("More than 1 param group in optimizer.")
        set_lrs = set([optimizer.param_groups[p_group]["lr"] for p_group in range(p_groups)])

        if len(set_lrs) == 1:
            LRs.append(optimizer.param_groups[0]["lr"])
        else:
            LRs.append(list(set_lrs))
        
    for batch in train_loader:    
        batch_images, batch_corners, batch_visibilities = batch
        batch_images = batch_images.to(device)

        gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
        gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

        pred_heatmaps, pred_pafs = model(batch_images)
        loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        
        # time_batch = datetime.now()
        # print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Train Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_epoch}")
    
    torch.cuda.empty_cache()
    
    time_train_epoch = datetime.now()
    train_losses.append(train_loss / len(train_loader))
    print(f"Trained Epoch {epoch+1}/100, Avg Train Loss for this Epoch: {train_loss / len(train_loader)}, it took: {time_train_epoch-start_time_epoch}")
     
    if epoch > 8 and (epoch+1) % validate_every_nth_epoch == 0: # Validate every nth epoch starting from epoch 10
        print(f"LRs so far at Epoch {epoch+1}: {LRs}")
        
        start_time_epoch_val = datetime.now()
        val_preds = [] # [[x1, y1, 2.0, x2, y2,..], [x1, y1,..],..]
        val_gt_gates = [] # [[[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..],..    ],..]
        val_gt_visibilities = []
        model.eval()

        with torch.no_grad():
            for batch in val_loader:
                start_time_batch_val = datetime.now()
                # print(f"Started Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, at {start_time_batch_val-start_time_epoch_val}")
                batch_images, batch_corners, batch_visibilities = batch
                batch_images = batch_images.to(device)
                pred_heatmaps, pred_pafs = model(batch_images)
                
                for i in range(len(batch_images)):
                    corner_candidates = find_corner_candidates(pred_heatmaps[i])
                    pred_gates = find_gates(corner_candidates, pred_pafs[i], is_train=False)

                    val_preds.append(pred_gates)
                    val_gt_gates.append(batch_corners[i])
                    val_gt_visibilities.append(batch_visibilities[i])

                # print(f"Finished Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, after {datetime.now() - start_time_batch_val}")
        
        torch.cuda.empty_cache()
                    
        print(f"Validation Epoch completed, now scoring function. It took {datetime.now() - start_time_epoch_val}")

        val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
        val_f1s.append(val_score)

        end_time_epoch_val = datetime.now()
        print(f"Validated Epoch {epoch+1}/100, Val F1 score: {val_score}, LR: {scheduler.get_last_lr()}, it took: {end_time_epoch_val - start_time_epoch_val}")

        scheduler.step(val_score)

        if early_stopper.early_stop(val_score, epoch, model):
            print(f"Early stopped at epoch {epoch+1}, best epoch: {early_stopper.best_epoch + 1}, best score: {early_stopper.best_score}")
            break

# Save info
best_score = early_stopper.best_score # Best F1 validation score due to early stopper
best_epoch = early_stopper.best_epoch # Epoch number at best F1 validation score (starts at 0)
best_model_path = f"/kaggle/working/final_model_enhanced_validation.pt"
print(f"Best model path: {best_model_path}, best epoch: {best_epoch}, best score: {best_score}, LRs: {LRs}")
print(f"For Plotting. Train losses per epoch: {train_losses}, validation F1 every {validate_every_nth_epoch}th epoch, starting at 10: {val_f1s}")

end_time_val = datetime.now()
time_elapsed_val = end_time_val - start_time
print(f"Time for full validation since start: {time_elapsed_val}")

valid_results = {
    "best_score": str(best_score),
    "best_epoch": str(best_epoch),
    "LRs": LRs,
    "best_model_path": best_model_path,
    "train_losses_per_epoch": train_losses,
    f"val_f1s_per_{validate_every_nth_epoch}th_epoch": val_f1s,
    "validation_time_since_start": str(time_elapsed_val)
}

with open("/kaggle/working/validation_results_enhanced.json", "w") as f:
    json.dump(valid_results, f, indent=4)
print("Validation results stored")
# To recover: json.load(f)


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Final time elapsed for validation: {time_elapsed}")
print(f"Total time: {total_time}")


# Training times: 
# Old version, 50/495 batches, batch size 16, non-vectorized, 187.090474
# Frame rate 20:
# 100/990 batches, batch size 8, vectorized, 103.936183. 2nd try: 105.857762 (200 batches 211.771808, /2 = 105.885904) --> 1 epoch = 17:28 min
# 50/495 batches, batch size 16, vectorized, 106.674084. 2nd try: 102.302779 (200 batches 205.060813, /2 = 102.5304065) --> 1 epoch = 16:55 min
# 50/495 batches, batch size 16, non-vectorized, 97.047515 (100 batches 194.828691, /2 = 97.4143) --> 1 epoch = 16:04 min
# 100/990 batches, batch size 8, non-vectorized, 100.488205 (200 batches 202.255757, /2 = 101.1278785) --> 1 epoch = 16:41 min
# 25/248 batches, batch size 32, vectorized, 100.093411 (50 batches 200.37641, /2 = 100.188205) --> 1 epoch = 16:34 min
# 25/248 batches, batch size 32, non-vectorized, 96.988917 (50 batches 193.104077, /2 = 96.5520385) --> 1 epoch = 15:58 min or 12:48 min for 80% of data (6:24 for frame rate 40)

# Test times:
# Frame rate 20:
# 50/495 batches, batch size 16, 94.798 (100 batches 186.508877, /2 = 93.2544) --> 1 loop through 20% of data = 3:05 min
# 100/990 batches, batch size 8, 89.206672 (200 batches 175.150122, /2 = 87.575061) --> 1 loop through 20% of data = 2:53 min (1:27 for frame rate 40)
# 800/7917 batches, batch size 1, 89.934446

# 50 epochs with frame rate 40: 320 (train) + 14:30 (val) = 5h, 35 min (6h, 33 min validating at every epoch). Training: 6h 40 min

In [ ]:
# # PLOT VALIDATION PROGRESS FOR SUBMISSION V2

# train_losses = 100*np.array([0.0247205510864606, 0.0032861434121496313, 0.0026578764421957553, 0.0024717147001019207, 0.0023988921559498747, 0.002334745774866257, 0.002286991761480965, 0.00221532485148141, 0.002061525521117567, 0.0018013109254770645, 0.0014850826286203643, 0.0012324318799217886, 0.001077970380360021, 0.0009667833271150541, 0.0008850138332455023, 0.0008313238147028381, 0.0007691430048970966, 0.0007460659011359335, 0.0007172849955438351, 0.0006849135426080573, 0.0006669951316965098, 0.0006367378170247014, 0.0006278900056155306, 0.0006221283076211146, 0.0005964727419322486, 0.0005793558643430029, 0.0005729219966710979, 0.0005679014340587107, 0.0005647690778270843, 0.0005457260470726703, 0.0005407993025307793, 0.000524134501367623, 0.0005146812213617055, 0.0005005333618154879, 0.0005065965123871102, 0.000502295734759623, 0.0004938324350900579, 0.0004936205773582073, 0.00048666866467389654, 0.00048009053133151467, 0.00046971950006883335, 0.0004729638230585779, 0.0004650076184454051, 0.0004605531112157214, 0.00045907406175582866, 0.0004507288518868903, 0.0004373287887538387, 0.0004388621822691126, 0.0004383838227788669, 0.0004352965354135647, 0.00042752584927752244, 0.0004346413053180285, 0.00042403010290696994, 0.0004259483490512697, 0.0004258995304013243, 0.0004096202247448745, 0.0003822401115822947, 0.0003777910453281227, 0.00037402629801579337, 0.0003714197128550103, 0.0003744806090973797, 0.00036734548117273885, 0.00036695768948907196, 0.00037385569568400707, 0.0003639708630715285, 0.0003483778830804392, 0.00034309743939885074, 0.00033905114033397766, 0.00034649093147819067])
# val_f1s = [0.19500949569918674, 0.433908474341642, 0.7195362739136993, 0.7577865799729769, 0.8080720644615201, 0.7822474939815057, 0.8245820642624395, 0.8184722964914666, 0.8551051343625895, 0.8214311704563148, 0.8862049147380443, 0.8445651736289346, 0.8439060817457011, 0.8930621057761512, 0.893950593977976, 0.9054106631874212, 0.8704540457986832, 0.9073186489368137, 0.9120817622290668, 0.9054911985955201, 0.8979063774075738, 0.9072036166618991, 0.9233711475099462, 0.9063861266819023, 0.9183406714484681, 0.9124834125107878, 0.907400532755843, 0.9252883318470403, 0.9124300852247421, 0.9003872639628195, 0.9243469540122083, 0.9303077191915501, 0.9179548264976616, 0.9305195341012326, 0.9237521652462999, 0.9228920973583803, 0.924572878841323, 0.9147255274107355, 0.9120661963778721, 0.9348991155536531, 0.9367698451217935, 0.9247195689690798, 0.9242638923251884, 0.9202581207627629, 0.9262972263245958, 0.9308246389559447, 0.9043321043190946, 0.9363128248194345, 0.9391486190783773, 0.9243865968233986, 0.9258873904692049, 0.9359035274389982, 0.9274615392198331, 0.9297547598016912, 0.9361368390859047, 0.9309446878325472, 0.930248000446712, 0.9313299721331172, 0.9373499273565423, 0.9335899027603357]

# epoch_plotting = list(range(1, len(train_losses)+1))
# validate_every_nth_epoch = 1

# plt.plot(epoch_plotting, train_losses, label='Train loss')
# # plt.yscale("log")
# plt.plot(epoch_plotting[9::validate_every_nth_epoch], val_f1s, label='Validation F1')
# plt.scatter(epoch_plotting[57], val_f1s[57-10], label="Best epoch")
# plt.legend()
# plt.show()

In [ ]:
# # 2nd BEST PRIVATE SCORE MODEL (V2 FINAL EPOCH, THRESHOLDS 0.15 AND 0.1)

# start_time = datetime.now()


# class EarlyStopper:
#     def __init__(self, patience=5, delta=0.001, best_model_path="/kaggle/working/final_model_enhanced_validation.pt"):
#         self.patience = patience
#         self.delta = delta
#         self.counter = 0
#         self.best_score = -float('inf')
#         self.best_epoch = -1
#         self.best_model_path = best_model_path

#     def early_stop(self, current_score, epoch, model):
#         if current_score > self.best_score + self.delta:
#             self.best_score = current_score
#             self.best_epoch = epoch
#             self.counter = 0
#             torch.save(model.state_dict(), self.best_model_path)
#             print(f"Saved model, best Epoch is {self.best_epoch + 1} with a score of {self.best_score}.")
#             # To recover: m = UNet(); m.load_state_dict(torch.load(path)); m.to(device); m.eval()
#             return False

#         self.counter += 1

#         if self.counter >= self.patience:
#             return True

#         return False


# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

# batch_size = 32
# frame_rate = 70 # About 2000 video samples

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)
# val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=frame_rate) # Different dataset for correct transforms

# compensated_indexes = np.array([i for i in range(train_ds.cum_sizes[5]) for _ in range(4)] + [i for i in range(train_ds.cum_sizes[5], len(train_ds))]) # Take non-video images 4 times, rest once
# updated_file_lengths = [(file_lengths[i]-1)//frame_rate + 1 if f.startswith(('autonomous', 'piloted')) else file_lengths[i]*4 for (i,f) in enumerate(filenames)]
# updated_video_ids = np.array([file_idx for file_idx, length in enumerate(updated_file_lengths) for _ in range(length)])

# current_train_size = 0
# goal_train_size = round(0.8*sum(updated_file_lengths))
# train_groups, val_groups = [], []

# np.random.seed(42)
# for g in np.random.choice(len(updated_file_lengths), len(updated_file_lengths), replace=False): # Choose random indexes
#     if current_train_size + updated_file_lengths[g] <= goal_train_size:
#         train_groups.append(g)
#         current_train_size += updated_file_lengths[g]
#     else:
#         val_groups.append(g)

# train_ids = np.concatenate([np.where(updated_video_ids == g)[0] for g in train_groups])
# val_ids = np.concatenate([np.where(updated_video_ids == g)[0] for g in val_groups])

# train_idxs = compensated_indexes[train_ids]
# val_idxs = compensated_indexes[val_ids]

# print(f"{len(train_idxs)} train files, {len(val_idxs)} validation files")

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# torch.manual_seed(42)
# train_sampler = torch.utils.data.SubsetRandomSampler(train_idxs)
# val_sampler = torch.utils.data.SubsetRandomSampler(val_idxs)

# train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, num_workers=4, collate_fn=collate_batch)
# val_loader = DataLoader(val_ds, batch_size=batch_size, sampler=val_sampler, num_workers=4, collate_fn=collate_batch)

# model = UNet()
# model.to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3) # Starting learning rate 1e-3
# scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True) # Reduce learning rate to half (factor), if validation score didn't increase in the last 3 times (patience) you did a validation, i.e. could be every 5*3 epochs. In case of not reaching minima.
# early_stopper = EarlyStopper(patience=11)
# scaler = torch.amp.GradScaler('cuda')

# validate_every_nth_epoch = 1

# train_losses = [] # For plotting
# val_f1s = [] # For plotting, at every 5th epoch
# LRs = [] # Save the learning rate for each epoch

# for epoch in range(100):
#     start_time_epoch = datetime.now()
#     print(f"Starting Epoch {epoch+1}/100.")
#     model.train()
#     train_loss = 0.0

#     p_groups = len(optimizer.param_groups)
#     if p_groups == 1:
#         LRs.append(optimizer.param_groups[0]["lr"]) # Save LRs
#     else:
#         print("More than 1 param group in optimizer.")
#         set_lrs = set([optimizer.param_groups[p_group]["lr"] for p_group in range(p_groups)])

#         if len(set_lrs) == 1:
#             LRs.append(optimizer.param_groups[0]["lr"])
#         else:
#             LRs.append(list(set_lrs))
        
#     for batch in train_loader:    
#         batch_images, batch_corners, batch_visibilities = batch
#         batch_images = batch_images.to(device)

#         gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
#         gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

#         pred_heatmaps, pred_pafs = model(batch_images)
#         loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         train_loss += loss.item()
        
#         # time_batch = datetime.now()
#         # print(f"Finished batch {i+1}/{len(train_loader)}, Epoch {epoch+1}/200, Train Loss: {train_loss/(i+1)}, Time: {time_batch-start_time_epoch}")
    
#     # torch.cuda.empty_cache()
    
#     time_train_epoch = datetime.now()
#     train_losses.append(train_loss / len(train_loader))
#     print(f"Trained Epoch {epoch+1}/100, Avg Train Loss for this Epoch: {train_loss / len(train_loader)}, it took: {time_train_epoch-start_time_epoch}")
     
#     if epoch > 8 and (epoch+1) % validate_every_nth_epoch == 0: # Validate every nth epoch starting from epoch 10
#         print(f"LRs so far at Epoch {epoch+1}: {LRs}")
        
#         start_time_epoch_val = datetime.now()
#         val_preds = [] # [[x1, y1, 2.0, x2, y2,..], [x1, y1,..],..]
#         val_gt_gates = [] # [[[[x1,y1], [x2,y2],..], [[x1,y1], [x2,y2],..],..    ],..]
#         val_gt_visibilities = []
#         model.eval()

#         with torch.no_grad():
#             for batch in val_loader:
#                 start_time_batch_val = datetime.now()
#                 # print(f"Started Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, at {start_time_batch_val-start_time_epoch_val}")
#                 batch_images, batch_corners, batch_visibilities = batch
#                 batch_images = batch_images.to(device)
#                 pred_heatmaps, pred_pafs = model(batch_images)
                
#                 for i in range(len(batch_images)):
#                     corner_candidates = find_corner_candidates(pred_heatmaps[i])
#                     pred_gates = find_gates(corner_candidates, pred_pafs[i], is_train=False)

#                     val_preds.append(pred_gates)
#                     val_gt_gates.append(batch_corners[i])
#                     val_gt_visibilities.append(batch_visibilities[i])

#                 # print(f"Finished Epoch {epoch+1}, batch {k+1}/{len(val_loader)}, after {datetime.now() - start_time_batch_val}")
        
#         # torch.cuda.empty_cache()
                    
#         print(f"Validation Epoch completed, now scoring function. It took {datetime.now() - start_time_epoch_val}")

#         val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
#         val_f1s.append(val_score)

#         end_time_epoch_val = datetime.now()
#         print(f"Validated Epoch {epoch+1}/100, Val F1 score: {val_score}, LR: {scheduler.get_last_lr()}, it took: {end_time_epoch_val - start_time_epoch_val}")

#         scheduler.step(val_score)

#         if early_stopper.early_stop(val_score, epoch, model):
#             print(f"Early stopped at epoch {epoch+1}, best epoch: {early_stopper.best_epoch + 1}, best score: {early_stopper.best_score}")
#             break

# # Save info
# best_score = early_stopper.best_score # Best F1 validation score due to early stopper
# best_epoch = early_stopper.best_epoch # Epoch number at best F1 validation score (starts at 0)
# best_model_path = f"/kaggle/working/final_model_enhanced_validation.pt"
# print(f"Best model path: {best_model_path}, best epoch: {best_epoch}, best score: {best_score}, LRs: {LRs}")
# print(f"For Plotting. Train losses per epoch: {train_losses}, validation F1 every {validate_every_nth_epoch}th epoch, starting at 10: {val_f1s}")

# end_time_val = datetime.now()
# time_elapsed_val = end_time_val - start_time
# print(f"Time for full validation since start: {time_elapsed_val}")

# valid_results = {
#     "best_score": str(best_score),
#     "best_epoch": str(best_epoch),
#     "LRs": LRs,
#     "best_model_path": best_model_path,
#     "train_losses_per_epoch": train_losses,
#     f"val_f1s_per_{validate_every_nth_epoch}th_epoch": val_f1s,
#     "validation_time_since_start": str(time_elapsed_val)
# }

# with open("/kaggle/working/validation_results_enhanced.json", "w") as f:
#     json.dump(valid_results, f, indent=4)
# print("Validation results stored")
# # To recover: json.load(f)


# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Final time elapsed for validation: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# FINAL TRAINING, SUBMISSION V2

start_time_train = datetime.now()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used: {device}")
scaler = torch.amp.GradScaler('cuda')

torch.manual_seed(42)

train_transforms = Compose([
    Rotate(limit=30, p=0.5),
    RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    GaussNoise(std_range=(0.2,0.3), p=0.3),
    Blur(blur_limit=3, p=0.2),
    Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
    ToTensorV2()
], keypoint_params={'format': 'xy', 'remove_invisible': False})

batch_size = 32
frame_rate = 90 # About 1500 video samples

train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)

compensated_indexes = np.array([i for i in range(train_ds.cum_sizes[5]) for _ in range(3)] + [i for i in range(train_ds.cum_sizes[5], len(train_ds))]) # Take non-video images 4 times, rest once


full_sampler = torch.utils.data.SubsetRandomSampler(compensated_indexes)
full_loader = DataLoader(train_ds, batch_size=batch_size, sampler=full_sampler, num_workers=4, collate_fn=collate_batch)

model = UNet()
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# LRs = [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.0005, 0.00025, 0.00025, 0.00025, 0.00025, 0.00025]
# LRs_best = LRs
LRs_best = LRs[:int(best_epoch + 1)]

final_train_losses = []

for epoch in range(69): 
    start_time_train_epoch = datetime.now()
    print(f"Starting epoch {epoch+1}/69, Time: {start_time_train_epoch - start_time_train}")
    model.train()
    train_loss = 0.0

    for param_group in optimizer.param_groups:
        param_group["lr"] = LRs_best[epoch]

    for batch in full_loader: 
        batch_images, batch_corners, batch_visibilities = batch
        batch_images = batch_images.to(device)

        gt_heatmaps = generate_gt_heatmaps(batch_corners, batch_visibilities, (240,320), device)
        gt_pafs = generate_gt_pafs(batch_corners, batch_visibilities, (240,320), device)

        pred_heatmaps, pred_pafs = model(batch_images)
        loss = loss_function(pred_heatmaps, gt_heatmaps, pred_pafs, gt_pafs)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        
        # time_batch = datetime.now()
        # print(f"Finished batch number {i+1}/{len(full_loader)}, Loss: {train_loss/(i+1)}, it took: {time_batch-start_time_train_epoch}")

    torch.cuda.empty_cache()

    print(f"Fully trained epoch {epoch+1}/69, Avg Train Loss: {train_loss / len(full_loader)}")
    final_train_losses.append(train_loss / len(full_loader))

    torch.save(model.state_dict(), f"/kaggle/working/current_final_model_enhanced.pt")
    print(f"Saved model at Epoch {epoch+1}.")
        

    # torch.save(model.state_dict(), "/kaggle/working/final_model.pt")
    # train_results = {
    #     "model_path": "/kaggle/working/final_model.pt",
    #     "train_losses_per_epoch": final_train_losses,
    #     "time_elapsed_train": str(datetime.now()-start_time_train)
    # }
    
    # with open("/kaggle/working/train_results.json", "w") as f:
    #     json.dump(train_results, f, indent=4)
    # print("Train results stored")


end_time_train = datetime.now()
time_elapsed_train = end_time_train - start_time_train
final_model_path = "/kaggle/working/final_model_enhanced.pt"
torch.save(model.state_dict(), final_model_path)

print(f"Time elapsed for training: {time_elapsed_train}")
print(f"Path for final model: {final_model_path}")
print(f"For plotting, losses: {final_train_losses}")


train_results = {
    "model_path": final_model_path,
    "time_elapsed_train": str(time_elapsed_train)
}

with open("/kaggle/working/train_results_enhanced.json", "w") as f:
    json.dump(train_results, f, indent=4)
print("All train results stored")

print("Training completed!")


end_time = datetime.now()
time_elapsed = end_time - start_time_train
total_time += time_elapsed

print(f"Time elapsed for training: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# # VISUALIZE SINGLE PREDICTIONS ON TEST DATA

# m = UNet() 
# # m.load_state_dict(torch.load("/kaggle/input/simple-validation-full-dataset/pytorch/default/1/final_model_simple_val.pt", map_location=torch.device('cpu')))
# m.load_state_dict(torch.load("/kaggle/input/simple-validation-full-dataset/pytorch/model_epoch_60_simple_val_full_dataset_v1/1/model_epoch_60_simple_val_full_dataset_v1.pt", map_location=torch.device('cpu')))
# m.to("cpu")
# m.eval()

# test_transforms = Compose([ToTensorV2()])
# test_ds = CompDataset(transforms=test_transforms, is_train=False, frame_rate=1)

# print(len(test_ds))

# image, _, _ = test_ds[5]

# with torch.no_grad():

#     pred_heatmaps, pred_pafs = m(torch.stack([image]))
    
#     corner_candidates = find_corner_candidates(pred_heatmaps[0])
#     pred_gates = find_gates(corner_candidates, pred_pafs[0], is_train=False)
#     print(pred_gates)
    
#     plt.imshow(image.numpy().transpose(1,2,0))
    
#     x_tl = pred_gates[::3*4]
#     x_tr = pred_gates[3::3*4]
#     x_br = pred_gates[6::3*4]
#     x_bl = pred_gates[9::3*4]
    
#     y_tl = pred_gates[1::3*4]
#     y_tr = pred_gates[4::3*4]
#     y_br = pred_gates[7::3*4]
#     y_bl = pred_gates[10::3*4]
    
#     visibilities = pred_gates[2::3]
    
#     plt.scatter(
#         x_tl,
#         y_tl,
#         c='yellow',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_tr,
#         y_tr,
#         c='red',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_br,
#         y_br,
#         c='blue',
#         marker='o',
#         s=10
#     )
    
#     plt.scatter(
#         x_bl,
#         y_bl,
#         c='white',
#         marker='o',
#         s=10
#     )
        
#     print(visibilities)
    
#     plt.show()
    
#     # TL = yellow, TR = red, BR = blue, BL = white

#     plt.subplots(2,2,figsize = (20,20))
#     for i in range(4):
#         plt.subplot(2,2,i+1)
#         plt.imshow(pred_heatmaps[0][i])
#     plt.show()

    
#     plt.subplots(2,2,figsize = (20,20))
#     paf_nums = [0, 3, 5, 6]
#     for i in range(4):
#         plt.subplot(2,2,i+1)
#         plt.imshow(pred_pafs[0][paf_nums[i]])
#     plt.show()

# # Yellow, red, blue, white

In [ ]:
# TUNE DETECTION THRESHOLDS

start_time = datetime.now()


def tune_detection_thresholds(model, val_loader, device, heat_range=(0.05, 0.55, 0.05), line_similarity_range=(0.05, 0.55, 0.05), patience_heat=5, patience_line=5, out_file="tuned_detection_thresholds.json"):
    start_time_thresholds = datetime.now()

    best_f1 = 0
    best_thresholds = [0, 0]

    heat_thresholds = np.arange(*heat_range).round(decimals=2)
    line_sim_thresholds = np.arange(*line_similarity_range).round(decimals=2)

    waiting_heat = 0

    model.eval()

    with torch.no_grad():
        for ht in heat_thresholds:
            waiting_line = 0
            
            for lt in line_sim_thresholds:
                val_preds, val_gt_gates, val_gt_visibilities = [], [], []

                print(f"Starting thresholds {[ht, lt]}/{[heat_range[1]-heat_range[2], line_similarity_range[1]-line_similarity_range[2]]} at time: {datetime.now() - start_time_thresholds}")

                for batch in val_loader:
                    batch_images, batch_corners, batch_visibilities = batch
                    batch_images = batch_images.to(device)
                    pred_heatmaps, pred_pafs = model(batch_images)
                    
                    for i in range(len(batch_images)):
                        corner_candidates = find_corner_candidates(pred_heatmaps[i], threshold=ht.item())
                        pred_gates = find_gates(corner_candidates, pred_pafs[i], sim_threshold=lt.item(), is_train=False)
    
                        val_preds.append(pred_gates)
                        val_gt_gates.append(batch_corners[i])
                        val_gt_visibilities.append(batch_visibilities[i])

                val_score = val_score_func(val_gt_gates, val_preds, val_gt_visibilities)  # Obtain avg F1 score
                
                if val_score > best_f1:
                    best_f1 = val_score
                    best_thresholds = [ht, lt]
                    waiting_line = 0
                    waiting_heat = 0
                    
                    print(f"New best score: {best_f1}, with thresholds: {best_thresholds}")

                    tuned_threshold_info = {
                        "best_f1_score": best_f1,
                        "best_tresholds": best_thresholds
                    }
                    
                    with open(f"/kaggle/working/{out_file}", "w") as f:
                        json.dump(tuned_threshold_info, f, indent=4)

                elif val_score == best_f1:
                    print(f"Thresholds {[ht, lt]} with score {val_score} are same as best.")
                    best_thresholds = best_thresholds + [ht, lt]

                else:       
                    print(f"Thresholds {[ht, lt]} with score {val_score} are not better than best.")
                    waiting_line += 1  

                    if waiting_line >= patience_line:
                        print("Breaked early from line similarity thresholds.")
                        break

            if waiting_heat >= patience_heat:
                print("Breaked early!")
                break
        
            waiting_heat += 1

    return best_thresholds, best_f1


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

# RESULTS ON FULL NON-VIDEO DATASET:

# For model simple_validation_full_dataset_epoch_60: 
# Thresholds: [0.5, 0.05, 0.5, 0.1, 0.5, 0.15, 0.5, 0.2, 0.5, 0.25, 0.5, 0.3, 0.5, 0.35, 0.5, 0.4, 0.5, 0.45, 0.5, 0.5], score 0.9421720240435867

# For model last_epoch_enhanced_validation:
# Thresholds: [0.15, 0.05, 0.15, 0.1, 0.15, 0.15, 0.15, 0.2, 0.15, 0.25, 0.15, 0.3, 0.15, 0.35, 0.15, 0.4, 0.15, 0.45], score 0.9563140353713259
# I.e. 0.15, 0.05-0.45
# --> 0.15, 0.1

# For model epoch_57_enhanced_validaion:
# Thresholds: [0.25, 0.05, 0.25, 0.1, 0.25, 0.15, 0.25, 0.2, 0.25, 0.25, 0.25, 0.3, 0.25, 0.35, 0.3, 0.05, 0.3, 0.1, 0.3, 0.15, 0.3, 0.2, 0.3, 0.25, 0.3, 0.3, 0.3, 0.35], score 0.9393513518628399
# I.e. 0.25, 0.05-0.35 and 0.3, 0.05-0.35
# --> 0.25, 0.05

# For model simple_validation_full_dataset_final_epoch:
# --> 0.15, 0.1

In [ ]:
# # GET VALIDATION SAMPLES FOR FINAL THRESHOLDS FOR MODELS

# train_transforms = Compose([
#     Rotate(limit=30, p=0.5),
#     RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
#     GaussNoise(std_range=(0.2,0.3), p=0.3),
#     Blur(blur_limit=3, p=0.2),
#     Affine(scale=(0.8, 1.2), translate_percent=0.1, shear=10, p=0.5),
#     ToTensorV2()
# ], keypoint_params={'format': 'xy', 'remove_invisible': False})

# batch_size = 32
# frame_rate = 70 # About 2000 video samples

# train_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=train_transforms, is_train=True, frame_rate=frame_rate)

# compensated_indexes = np.array([i for i in range(train_ds.cum_sizes[5]) for _ in range(4)] + [i for i in range(train_ds.cum_sizes[5], len(train_ds))]) # Take non-video images 4 times, rest once
# updated_file_lengths = [(file_lengths[i]-1)//frame_rate + 1 if f.startswith(('autonomous', 'piloted')) else file_lengths[i]*4 for (i,f) in enumerate(filenames)]
# updated_video_ids = np.array([file_idx for file_idx, length in enumerate(updated_file_lengths) for _ in range(length)])

# current_train_size = 0
# goal_train_size = round(0.8*sum(updated_file_lengths))
# train_groups, val_groups = [], []

# np.random.seed(42)
# for g in np.random.choice(len(updated_file_lengths), len(updated_file_lengths), replace=False): # Choose random indexes
#     if current_train_size + updated_file_lengths[g] <= goal_train_size:
#         train_groups.append(g)
#         current_train_size += updated_file_lengths[g]
#     else:
#         val_groups.append(g)

# val_ids = np.concatenate([np.where(updated_video_ids == g)[0] for g in val_groups])

# val_idxs = compensated_indexes[val_ids]

# threshold_val_indexes = np.array(list(set(val_idxs[val_idxs<=498])))

# torch.manual_seed(42)


In [ ]:
# # FIND FINAL THRESHOLDS FOR MODELS

# start_time = datetime.now()

# print(f"{len(threshold_val_indexes)} validation files")

# val_transforms = Compose([ToTensorV2()], keypoint_params={'format': 'xy', 'remove_invisible': False})

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")

# batch_size = 16
# frame_rate = 70 # About 2000 video samples

# val_ds = CompDataset(file_lengths=file_lengths, filenames=filenames, transforms=val_transforms, is_train=True, frame_rate=frame_rate) # Different dataset for correct transforms
# # val_idxs = list(range(sum(file_lengths[:6]))) # Only use non-video images, all
# # print(f"{len(val_idxs)} validation files")


# # val_subset = torch.utils.data.Subset(val_ds, val_idxs)
# val_subset = torch.utils.data.Subset(val_ds, threshold_val_indexes)
# val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_batch)

# model1 = UNet() 
# final_model_path1 = "/kaggle/input/final-model-enhanced-validation-epoch-57/pytorch/default/1/final_model_enhanced_epoch_57.pt"
# model1.to(device)
# model1.load_state_dict(torch.load(final_model_path1, map_location=device))

# final_thresholds1, threshold_scores1 = tune_detection_thresholds(model1, val_loader, device, heat_range=(0.05, 0.55, 0.05), line_similarity_range=(0.05, 0.55, 0.05), patience_heat=5, patience_line=5, out_file = "tuned_detection_thresholds_1.json")
# print(f"For model 1, final thresholds are {final_thresholds1} with score {threshold_scores1}")

# end_time = datetime.now()
# time_elapsed = end_time - start_time
# total_time += time_elapsed

# print(f"Time elapsed for finding the thresholds: {time_elapsed}")
# print(f"Total time: {total_time}")

In [ ]:
# TEST PREDICTION WITH TTA

start_time = datetime.now()


def rotate_paf_vectors(pafs, theta_deg):
    theta_rad = torch.deg2rad(torch.tensor(theta_deg))
    cos_theta = torch.cos(theta_rad)
    sin_theta = torch.sin(theta_rad)
    rotated_pafs = pafs.clone()

    for ch in range(4):
        x_components = pafs[2*ch]
        y_components = pafs[2*ch + 1]

        rotated_pafs[2*ch] = x_components * cos_theta - y_components * sin_theta
        rotated_pafs[2*ch + 1] = x_components * sin_theta + y_components * cos_theta
    return rotated_pafs


def predict_final_gates(model, image, device, heat_threshold, line_sim_threshold, TTA=True):
    with torch.no_grad():
        heatmaps, pafs = model(image.unsqueeze(0))

        if TTA:
            heatmap_list, paf_list = [], []
            heatmap_list.append(heatmaps[0])
            paf_list.append(pafs[0])

            augmentations = [lambda x: VF.rotate(x, 15),  # +15 degrees
                             lambda x: VF.rotate(x, -15)]  # -15 degrees
            inv_augmentations = [lambda x: (VF.rotate(x, -15), -15.0), # Rotate image and also PAF vectors
                                 lambda x: (VF.rotate(x, 15), 15.0)]

            for aug_func, inv_func in zip(augmentations, inv_augmentations):
                aug_img = aug_func(image.to(device)).unsqueeze(0)
                heatmaps, pafs = model(aug_img)
                heatmap_list.append(inv_func(heatmaps[0])[0])
                inv_pafs, angle = inv_func(pafs[0])
                paf_list.append(rotate_paf_vectors(inv_pafs, angle))

            pred_heatmaps = torch.mean(torch.stack(heatmap_list), dim=0)
            pred_pafs = torch.mean(torch.stack(paf_list), dim=0)

        else:
            pred_heatmaps, pred_pafs = heatmaps[0], pafs[0]

    corner_candidates = find_corner_candidates(pred_heatmaps, threshold=heat_threshold)
    pred_gates = find_gates(corner_candidates, pred_pafs, sim_threshold=line_sim_threshold, is_train=False)
    return pred_gates


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# PREDICTIONS TO CSV FORMAT

start_time = datetime.now()


def csv_output(all_gates, filename='submission.csv'):
    non_valid = [str(100/320.0), str(60/240.0), str(2.0), str(220/320.0), str(60/240.0), str(2.0), str(220/320.0), str(180/240.0), str(2.0), str(100/320.0), str(180/240.0), str(2.0)]
    # all_gates = [[x1, y1, 2.0, x2, y2, 2.0,.., x5, y5, 2.0,..], [x1, y1, 2.0,..],..], i.e. all images
    output = [' '.join([str(v/320.0) if i%3 == 0 else str(v/240.0) if i%3 == 1 else str(v) for i,v in enumerate(g)]) if len(g)>0 else ' '.join(non_valid) for g in all_gates]
    
    data = {"Id": range(len(all_gates)), "PredictionString": output}
    df = pd.DataFrame(data=data)
    
    df.to_csv('/kaggle/working/' + filename, index=False)
    return df

# After: submission = pd.read_csv("submission.csv")


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed: {time_elapsed}")
print(f"Total time: {total_time}")

In [ ]:
# FINAL TESTING

start_time = datetime.now()


# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device used: {device}")
# scaler = torch.amp.GradScaler('cuda')

test_transforms = Compose([ToTensorV2()])
test_ds = CompDataset(transforms=test_transforms, is_train=False, frame_rate=1)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, collate_fn=collate_batch)
print(f"Test files: {len(test_ds)}")

# model = UNet() 
# final_model_path = "/kaggle/input/final-model-enhanced-full-dataset/pytorch/default/1/final_model_enhanced_full_dataset.pt"
# model.to(device)
# model.load_state_dict(torch.load(final_model_path))

model.eval()

final_gate_predictions = []

for k, batch in enumerate(test_loader):
    print(f"Started batch {k+1}/{len(test_loader)} at time {datetime.now()-start_time}")

    batch_images, _, _ = batch
    batch_images = batch_images.to(device)

    for im in batch_images:
        pred_gates = predict_final_gates(model, im, device, heat_threshold=0.15, line_sim_threshold=0.1, TTA=True)
        final_gate_predictions.append(pred_gates)

final_predictions_df = csv_output(final_gate_predictions, filename="submission.csv")


end_time = datetime.now()
time_elapsed = end_time - start_time
total_time += time_elapsed

print(f"Time elapsed for testing: {time_elapsed}")
print(f"FINAL TIME: {total_time}")

In [ ]:
# # VISUALIZE PREDICTIONS ON TEST DATA

# test_transforms = Compose([ToTensorV2()])
# test_ds = CompDataset(transforms=test_transforms, is_train=False, frame_rate=1)

# # img_num = 25

# final_gate_predictions1 = pd.read_csv("/kaggle/input/final-submissions/submission_enhanced_validation_thresholds_015_01.csv")["PredictionString"]
# final_gate_predictions2 = pd.read_csv("/kaggle/input/final-submissions/submission_final_enhanced.csv")["PredictionString"]
# final_gate_predictions3 = pd.read_csv("/kaggle/input/final-submissions/submission_simple_val_full_dataset_final_epoch.csv")["PredictionString"]

# start_img = 40

# # for img_num in range(start_img, start_img+10):
# # for img_num in hard_images:
# for img_num in non_eq[start_img:start_img+10]:

#     image, _, _ = test_ds[img_num]
    
#     with torch.no_grad():
        
#         pred_gates1 = np.array(final_gate_predictions1[img_num].split(), dtype=float)
#         pred_gates2 = np.array(final_gate_predictions2[img_num].split(), dtype=float)
#         pred_gates3 = np.array(final_gate_predictions3[img_num].split(), dtype=float)
        
#         plt.imshow(image.numpy().transpose(1,2,0))
        
#         x_tl = np.round(pred_gates1[::3*4]*320)
#         x_tr = np.round(pred_gates1[3::3*4]*320)
#         x_br = np.round(pred_gates1[6::3*4]*320)
#         x_bl = np.round(pred_gates1[9::3*4]*320)
        
#         y_tl = np.round(pred_gates1[1::3*4]*240)
#         y_tr = np.round(pred_gates1[4::3*4]*240)
#         y_br = np.round(pred_gates1[7::3*4]*240)
#         y_bl = np.round(pred_gates1[10::3*4]*240)

#         visibilities1 = pred_gates1[2::3]
        
#         plt.scatter(
#             x_tl,
#             y_tl,
#             c='yellow',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_tr,
#             y_tr,
#             c='red',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_br,
#             y_br,
#             c='blue',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_bl,
#             y_bl,
#             c='white',
#             marker='o',
#             s=10
#         )
    
#         plt.show()

#         print(f"Submission 1 (full dataset) gates:\n{pred_gates1}")
#         print(f"Visibilities: {visibilities1}")
        
#         plt.imshow(image.numpy().transpose(1,2,0))
        
#         x_tl = np.round(pred_gates2[::3*4]*320)
#         x_tr = np.round(pred_gates2[3::3*4]*320)
#         x_br = np.round(pred_gates2[6::3*4]*320)
#         x_bl = np.round(pred_gates2[9::3*4]*320)
        
#         y_tl = np.round(pred_gates2[1::3*4]*240)
#         y_tr = np.round(pred_gates2[4::3*4]*240)
#         y_br = np.round(pred_gates2[7::3*4]*240)
#         y_bl = np.round(pred_gates2[10::3*4]*240)
        
#         visibilities2 = pred_gates2[2::3]
        
#         plt.scatter(
#             x_tl,
#             y_tl,
#             c='yellow',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_tr,
#             y_tr,
#             c='red',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_br,
#             y_br,
#             c='blue',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_bl,
#             y_bl,
#             c='white',
#             marker='o',
#             s=10
#         )
    
#         plt.show()

#         print(f"Submission 2 (over-sampled non-videos) gates:\n{pred_gates2}")
#         print(f"Visibilities: {visibilities2}")
    
#         plt.imshow(image.numpy().transpose(1,2,0))
        
#         x_tl = np.round(pred_gates3[::3*4]*320)
#         x_tr = np.round(pred_gates3[3::3*4]*320)
#         x_br = np.round(pred_gates3[6::3*4]*320)
#         x_bl = np.round(pred_gates3[9::3*4]*320)
        
#         y_tl = np.round(pred_gates3[1::3*4]*240)
#         y_tr = np.round(pred_gates3[4::3*4]*240)
#         y_br = np.round(pred_gates3[7::3*4]*240)
#         y_bl = np.round(pred_gates3[10::3*4]*240)
        
#         visibilities3 = pred_gates3[2::3]
        
#         plt.scatter(
#             x_tl,
#             y_tl,
#             c='yellow',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_tr,
#             y_tr,
#             c='red',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_br,
#             y_br,
#             c='blue',
#             marker='o',
#             s=10
#         )
        
#         plt.scatter(
#             x_bl,
#             y_bl,
#             c='white',
#             marker='o',
#             s=10
#         )
        
#         plt.show()

#         print(f"Submission 3 (only validated on non-videos) gates:\n{pred_gates3}")
#         print(f"Visibilities: {visibilities3}")
        
    
#         # plt.subplots(2,2,figsize = (20,20))
#         # for i in range(4):
#         #     plt.subplot(2,2,i+1)
#         #     plt.imshow(pred_heatmaps[0][i])
#         # plt.show()
    
        
#         # plt.subplots(2,2,figsize = (20,20))
#         # paf_nums = [0, 3, 5, 6]
#         # for i in range(4):
#         #     plt.subplot(2,2,i+1)
#         #     plt.imshow(pred_pafs[0][paf_nums[i]])
#         # plt.show()
    
#     # TL = yellow, TR = red, BR = blue, BL = white

# # Thresholds (model simple validation full dataset): 
# # 1: 0.3, 0.1 (df)
# # 2: 0.3, 0.3 (submission.csv)
# # 3: 0.2, 0.2
# # 4: 0.2, 0.15
# # 5: 0.15, 0.15
# # 6: 0.15, 0.1
# # 7: 0.5, 0.05
# # 8: 0.5, 0.1

# # Thresholds (model enhanced validation): 
# # 1: 0.3, 0.1 (df)
# # 2: 0.15, 0.05
# # 3: 0.15, 0.1
# # 4: 0.15, 0.15

# hard_images = [2, 5, 10, 11, 12, 14, 17, 18, 19, 23, 26, 27, 28, 51, 52, 55, 57, 58, 59] # As identified using model simple validation full dataset with 0.5, 0.05